# Twelve findings, four branches, one header pass

### A restructured 0.936 inference pipeline — and a harness for deciding what to change next

The score here is **macro ROC-AUC over twelve binary findings**, on a hidden test
split, in a nine-hour offline kernel. That one sentence decides most of what follows.

Under ROC-AUC only the *ordering inside each column* is scored. A monotone transform
of a column changes nothing: rank, sigmoid, calibration, clipping — all free, all
invisible. Two things do move the number:

1. **Which study sits above which** — the ensemble and its weights.
2. **Studies that could not be scored at all.** They get a constant, and a block of
   tied scores earns half credit against every study it ties with. A tie block is the
   most expensive thing in a pipeline like this and the easiest one to miss.

Both are measured explicitly here rather than assumed.

---

## What this notebook is

An inference pipeline fusing four independently-trained branches:

| Branch | Model | Preprocessing | Role in the blend |
|---|---|---|---|
| **A** | DINOv2-S/14 + per-finding slot attention | 6 slots (plane x contrast x fat-sat), 336 px, 130 mm crop | base vote |
| **B** | timm fold ensemble | 6 slots, 16 slices, 336 px | 0.45 against A |
| **C** | RadImageNet ResNet-50 + attention heads, three slot views | 224 px, 8 slices per plane | 0.50 on ten findings, then a 0.15 second pass, then an 88-feature calibrator |
| **D** | CoAtNet-RMLP-2 over 3-slice windows | 5 slots, 64 slices, 140 mm crop, 62 windows | ~0.50 per finding, correlation-gated |

The weights and the fusion arithmetic come from the upstream public checkpoint and
reproduce **0.936**. Full credit and licensing at the bottom: the models are not
mine, the pipeline around them is.

## What is different here

**One header pass instead of four.** The store holds roughly 800k slice headers. The
original walked the tree four times and read every header at least three times: once
for branch A's slice ordering, once per branch C cache build (three of those), and
again inside branch D. Cell 2 walks once, reads each header once into a typed index,
and derives all four slice-ordering dialects — normal, dominant-axis, InstanceNumber,
position — from that single pass.

**One decode sweep where the views overlap.** Branch C's three slot views share the
same fat-saturated series with the same band and slice count. `build_caches` takes a
*list* of specs and fills whichever fit in the memory budget together, so a slice is
decoded once instead of three times.

**Dead work removed.** Branch A used to compute a jittered prediction, a frontier
vote and a five-fold soft blend, then overwrite `submission.csv` with the frontier
vote and `unlink()` the other two. Only the surviving path is computed.

**No mutating globals.** `adopt_config_globals`, three `globals().update(...)` calls
and a snapshot/restore of the entire global namespace are replaced by an immutable
`PixelSpec` handed to the builders. Cell order no longer changes what a model
computes.

**A way to decide the next change.** Every blend weight lives in one `TUNING` dict,
and `RSNA_MODE=validate` reruns the pipeline on the gold-labelled studies and scores
it offline with bootstrap win-rates and weight sweeps (cell 13). The leaderboard is a
scarce oracle; this is how to stop spending it.

## How to read the rest

Cells 1-3 are infrastructure (config, shared index, volume construction). Cells 4-6
are model definitions. Cells 7-10 are the four branches. Cell 11 fuses. Cell 12
reports coverage, tie blocks and branch agreement. Cell 13 validates offline.

To experiment: change one entry in `TUNING` or `FLAGS` in cell 1, run with
`RSNA_MODE=validate`, and check the bootstrap win-rate before spending a submission.


## 1. Configuration

Paths, the twelve findings, and every blend weight in the pipeline collected into one
`TUNING` dict. `FLAGS` holds candidate levers that default to the 0.936 behaviour, so
a variant is a one-line edit rather than a hunt through 3,000 lines. `MODE=validate`
retargets the whole pipeline at the gold-labelled studies.

In [ ]:
# ============================================================================
# 1. Configuration
# ----------------------------------------------------------------------------
# Everything the four branches used to mutate through `globals()` lives here as
# an explicit, immutable object. Nothing below this cell writes to a global.
# ============================================================================
from __future__ import annotations

import contextlib
import gc
import glob
import hashlib
import json
import os
import re
import threading
import time
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field, replace
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F

os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
cv2.setNumThreads(1)
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

T0 = time.time()


def log(msg):
    print(f"[{time.time() - T0:7.1f}s] {msg}", flush=True)


class WeightsError(RuntimeError):
    pass


# ---------------------------------------------------------------- paths -----
def _competition_root():
    # API-attached competitions mount under /competitions/<slug>, UI-added ones
    # under the short path. Resolve both rather than assuming either.
    for candidate in (
        "/kaggle/input/competitions/rsna-knee-abnormality-detection",
        "/kaggle/input/rsna-knee-abnormality-detection",
    ):
        if os.path.isdir(candidate):
            return Path(candidate)
    raise RuntimeError("competition data not found under /kaggle/input")


ROOT = _competition_root()
WORK = Path("/kaggle/working")
ASSET = Path("/kaggle/input/rsna-knee-bend-dinov3-0917-repro-assets")
DINO_DIR = Path("/kaggle/input/models/metaresearch/dinov2/pytorch/small/1")
DINO_PACKAGE = ASSET / "rsna-knee-weights"
FOLD_CKPT_DIR = ASSET / "knee-mri-fold-weights"
RAD_ENCODER = ASSET / "resnet-50-radimagenet-marwan/ResNet50.pt"
RAD_HEADS_V15 = ASSET / "rsna-knee-e9-radimagenet-heads-v15/v52_radimagenet_heads.pt"
RAD_HEADS_E13 = ASSET / "kernel-sources/rsna-knee-e13-train/rsna_rad_e11/v52_e11_heads.pt"

SPLIT = "test_series" if (ROOT / "test_series").is_dir() else "train_series"
DEVICES = [torch.device(f"cuda:{i}") for i in range(torch.cuda.device_count())] or [
    torch.device("cpu")
]

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
    "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture",
]
SEED = 2026
TIME_BUDGET = 8.0 * 3600
HDR_THREADS = 32
PIX_THREADS = 12
CACHE_FRACTION = 0.45  # share of MemAvailable a single decode sweep may claim

# ------------------------------------------------------------------ mode ---
# "submit"   : score the hidden test split and write submission.csv
# "validate" : score only the studies that carry gold labels, so an ensemble
#              change can be judged offline instead of on the leaderboard.
MODE = os.environ.get("RSNA_MODE", "submit")
GOLD = None

if MODE == "validate":
    SPLIT = "train_series"
    _train = pd.read_csv(ROOT / "train.csv", dtype={"StudyInstanceUID": str})
    _labelled = _train.dropna(subset=[t for t in TARGETS if t in _train.columns])
    if not len(_labelled) or not set(TARGETS).issubset(_train.columns):
        raise RuntimeError(
            "validate mode needs train.csv to carry the twelve finding columns; "
            f"found {[c for c in _train.columns if c != 'StudyInstanceUID'][:6]}"
        )
    TEST = _labelled[["StudyInstanceUID"]].reset_index(drop=True)
    GOLD = _labelled.set_index("StudyInstanceUID")[TARGETS].astype(float)
    log(f"VALIDATE mode: {len(TEST)} gold-labelled studies")
else:
    TEST = pd.read_csv(ROOT / "test.csv", dtype={"StudyInstanceUID": str})

# --------------------------------------------------------- competition csv --
TEST_IDS = TEST["StudyInstanceUID"].astype(str).tolist()
SERIES_CSV = pd.read_csv(
    ROOT / f"{SPLIT}.csv",
    dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str},
)
SERIES_CSV = SERIES_CSV.loc[:, ~SERIES_CSV.columns.duplicated()]
PLANE_OF = dict(zip(SERIES_CSV.SeriesInstanceUID, SERIES_CSV.Anatomical_Plane))
SERIES_ROWS = {
    study: group.to_dict("records")
    for study, group in SERIES_CSV[SERIES_CSV.StudyInstanceUID.isin(set(TEST_IDS))]
    .groupby("StudyInstanceUID")
}

_sample = ROOT / "sample_submission.csv"
SUB_COLS = (
    list(pd.read_csv(_sample, nrows=1).columns)
    if _sample.is_file()
    else ["StudyInstanceUID"] + TARGETS
)
assert SUB_COLS[1:] == TARGETS, f"unexpected submission schema: {SUB_COLS}"
log(f"root {ROOT} | {len(TEST_IDS)} test studies | {len(SERIES_CSV)} series rows")
log(f"devices: {[str(d) for d in DEVICES]}")

# ------------------------------------------------------------ pixel specs --
# `rules` selects between the two preprocessing dialects the checkpoints were
# fitted under. Only these two combinations are reproducible.
RULES_NATIVE = {"order": "normal", "lat": "centre", "slot_fallback": False, "decode_fill": "nearest"}
RULES_LEGACY = {"order": "dominant_axis", "lat": "corner_x", "slot_fallback": True, "decode_fill": "zero"}

SLOTS_RECOVERED = (
    ("SAG_FLUID_FS", "Sagittal", True, True),
    ("COR_FLUID_FS", "Coronal", True, True),
    ("AX_FLUID_FS", "Axial", True, True),
    ("SAG_FLUID_NOFS", "Sagittal", True, False),
    ("COR_T1", "Coronal", False, False),
    ("SAG_T1", "Sagittal", False, False),
)
SLOTS_RAD_E10 = (
    ("SAG_FS", "Sagittal", None, True),
    ("COR_FS", "Coronal", None, True),
    ("AX_FS", "Axial", None, True),
)
SLOTS_RAD_E13 = SLOTS_RAD_E10 + (("SAG_NOFS", "Sagittal", None, False),)
SLOTS_RAD_E11 = (
    ("SAG_NOFS", "Sagittal", None, False),
    ("COR_NOFS", "Coronal", None, False),
    ("AX_NOFS", "Axial", None, False),
    ("SAG_FS", "Sagittal", None, True),
)

LAT_MIN_OFFSET_MM = 20.0
LEGACY_LAT_OFFSET_MM = 5.0


@dataclass(frozen=True)
class PixelSpec:
    """One decoding contract. Replaces the mutated globals of the original."""

    name: str
    slots: tuple
    img: int = 336
    slices: int = 3
    group: int = 3                 # window depth the head was fitted with
    crop_mm: float = 130.0
    band: tuple = (0.2, 0.8)
    rules: str = "native"          # "native" | "legacy"
    mode: str = "rescale"          # pixel decode dialect, see decode_slice()

    @property
    def rule(self):
        return RULES_NATIVE if self.rules == "native" else RULES_LEGACY

    @property
    def n_slot(self):
        return len(self.slots)

    @property
    def nbytes_per_study(self):
        return self.n_slot * self.slices * self.img * self.img


# ------------------------------------------------------------ rank helpers --
def rank_avg(values):
    """pandas average-tie percentile rank (branches A, C and the final blends)."""
    return (
        pd.DataFrame(np.asarray(values, np.float64))
        .rank(method="average", pct=True)
        .to_numpy(np.float64)
    )


def rank_ordinal(values):
    """argsort ordinal percentile rank (branch B fold mean, branch D windows)."""
    values = np.asarray(values)
    order = values.argsort(0).argsort(0).astype(np.float64)
    return order / max(1, values.shape[0] - 1)


def align_to_test(values, ids, fill=0.5):
    """Map a (studies x targets) matrix onto the submission row order."""
    position = {str(uid): i for i, uid in enumerate(ids)}
    out = np.full((len(TEST_IDS), len(TARGETS)), float(fill), np.float64)
    take = [position.get(uid) for uid in TEST_IDS]
    hit = np.array([i for i, p in enumerate(take) if p is not None], np.int64)
    src = np.array([p for p in take if p is not None], np.int64)
    if len(hit):
        out[hit] = np.asarray(values, np.float64)[src]
    missing = len(TEST_IDS) - len(hit)
    if missing:
        log(f"align_to_test: {missing} study(ies) absent, filled with {fill}")
    return out


def write_submission(values, path=WORK / "submission.csv", tag=""):
    values = np.asarray(values, np.float64)
    if values.shape != (len(TEST_IDS), len(TARGETS)):
        raise RuntimeError(f"submission shape {values.shape}")
    if not np.isfinite(values).all() or values.min() < 0 or values.max() > 1:
        raise RuntimeError("invalid submission values")
    frame = pd.DataFrame(values.astype(np.float32), columns=TARGETS)
    frame.insert(0, "StudyInstanceUID", TEST_IDS)
    frame = frame[SUB_COLS]
    frame.to_csv(path, index=False)
    log(f"wrote {path.name}{(' (' + tag + ')') if tag else ''}; {frame.shape}")
    return frame


def available_gb():
    try:
        with open("/proc/meminfo") as handle:
            info = {k.strip(): v for k, v in (l.split(":", 1) for l in handle if ":" in l)}
        return int(info["MemAvailable"].split()[0]) / 1024 ** 2
    except Exception:
        return 24.0


# --------------------------------------------------------------- knobs -----
# Every blend weight in the pipeline, in one place, so a variant is a one-line
# edit and `validate` mode can measure it. The defaults reproduce 0.936.
TUNING = {
    "fold_blend_w": 0.45,        # branch B against branch A
    "rad_alpha": 0.50,           # RadImageNet reference vote
    "rad_exclude": ("Baker's", "Fracture"),
    "rad_e13_w": 0.50,           # e13 view inside the reference vote
    "second_alpha": 0.15,        # e11 second-pass vote
    "cal_w": 0.40,               # 88-feature calibrator on gated findings
    "coatnet_base": 0.50,
    "coatnet_overrides": {"ACL": 0.53, "Medial Meniscus": 0.56,
                          "Lateral Meniscus": 0.61, "Lateral OA": 0.55,
                          "Fracture": 0.61},
    "coatnet_fallback": {"Medial Meniscus": 0.52, "Lateral Meniscus": 0.54,
                         "Fracture": 0.54},
    "raptor_complement": {"MCL": 0.16, "Medial OA": 0.10, "PF OA": 0.12,
                          "Effusion": 0.10, "Synovitis": 0.16, "Baker's": 0.12,
                          "Contusion": 0.12},
}

# Candidate score levers. All default to the 0.936 behaviour; switch one on,
# run `validate`, and keep it only if the bootstrap says so.
FLAGS = {
    "tta_jitter": False,       # branch A: average a mildly augmented second view
    "coverage_repair": False,  # fill an empty slot from another series of the same plane
}
AUG_ROT_DEG, AUG_SCALE, AUG_SHIFT, AUG_INTENSITY = 8.0, 0.08, 0.05, 0.1

# Branch outputs land here as (studies x targets) rank matrices in test order.
STAGE = {}

## 2. One header pass over the DICOM store

Every ordering rule the four branches use is a function of four header fields:
ImagePositionPatient, ImageOrientationPatient, InstanceNumber and PixelSpacing. Read
them once and the ordering dialects become pure functions over arrays.

Why four dialects and not one: each branch's weights were fitted against a particular
slice order, and a stack ordered differently is a different input. They are kept
distinct deliberately.

In [ ]:
# ============================================================================
# 2. One DICOM header pass for the whole run
# ----------------------------------------------------------------------------
# The original read the header of every slice at least three times (branch A's
# ordering pass, branch C's three cache builds, branch D's order_and_meta) and
# walked the tree four times. Here the tree is walked once, every slice header
# is read once, and the four different slice-ordering dialects are derived from
# that one in-memory index.
# ============================================================================
FILE_TAGS = ["ImagePositionPatient", "ImageOrientationPatient", "InstanceNumber", "PixelSpacing"]
SERIES_TAGS = [
    "SeriesDescription", "SequenceName", "ScanOptions", "ScanningSequence",
    "RepetitionTime", "EchoTime", "Laterality", "PixelSpacing", "Rows", "Columns",
    "RescaleSlope", "RescaleIntercept", "ImagePositionPatient", "ImageOrientationPatient",
]


@dataclass
class Series:
    study: str
    uid: str
    directory: str
    names: list                 # os.scandir order (branch B/D read in this order)
    sorted_names: list          # sorted order (branches A/C)
    ipp: np.ndarray             # (n, 3)
    iop: np.ndarray             # (n, 6)
    inst: np.ndarray            # (n,)
    spacing: np.ndarray         # (n,)
    tags: dict                  # middle-of-sorted-stack header, as in the original probe
    _orders: dict = field(default_factory=dict)

    def path(self, name):
        return os.path.join(self.directory, name)

    def order(self, rule):
        """Ordered file names under one dialect; computed once, cached."""
        if rule not in self._orders:
            self._orders[rule] = _ORDER_RULES[rule](self)
        return self._orders[rule]


def _as_vec(value, n):
    if value is None:
        return None
    try:
        vector = np.asarray([float(x) for x in value], np.float64)
    except (TypeError, ValueError):
        return None
    return vector[:n] if len(vector) >= n and np.isfinite(vector[:n]).all() else None


def _scan_series(item):
    study, uid, directory = item
    names = [entry.name for entry in os.scandir(directory) if entry.name.endswith(".dcm")]
    n = len(names)
    series = Series(
        study=study, uid=uid, directory=directory, names=names,
        sorted_names=sorted(names),
        ipp=np.full((n, 3), np.nan), iop=np.full((n, 6), np.nan),
        inst=np.full(n, np.nan), spacing=np.full(n, np.nan), tags={},
    )
    if n == 0:
        return series
    for i, name in enumerate(names):
        try:
            header = pydicom.dcmread(
                os.path.join(directory, name), stop_before_pixels=True,
                force=True, specific_tags=FILE_TAGS,
            )
        except Exception:
            continue
        vector = _as_vec(getattr(header, "ImagePositionPatient", None), 3)
        if vector is not None:
            series.ipp[i] = vector
        vector = _as_vec(getattr(header, "ImageOrientationPatient", None), 6)
        if vector is not None:
            series.iop[i] = vector
        number = getattr(header, "InstanceNumber", None)
        if number is not None:
            with contextlib.suppress(TypeError, ValueError):
                series.inst[i] = float(number)
        vector = _as_vec(getattr(header, "PixelSpacing", None), 1)
        if vector is not None:
            series.spacing[i] = vector[0]
    # Series-level metadata comes from the middle slice, exactly as before.
    try:
        header = pydicom.dcmread(
            series.path(series.sorted_names[n // 2]), stop_before_pixels=True, force=True
        )
        for tag in SERIES_TAGS:
            value = getattr(header, tag, None)
            if value is None:
                series.tags[tag] = None
            elif isinstance(value, (list, tuple)) or type(value).__name__ == "MultiValue":
                series.tags[tag] = "|".join(str(x) for x in value)
            else:
                series.tags[tag] = str(value)
    except Exception as exc:
        series.tags = {tag: None for tag in SERIES_TAGS}
        series.tags["err"] = str(exc)[:120]
    return series


def build_index(split=SPLIT, threads=HDR_THREADS):
    base = ROOT / split
    items = [
        (study.name, series.name, series.path)
        for study in os.scandir(base) if study.is_dir()
        for series in os.scandir(study.path) if series.is_dir()
    ]
    started = time.time()
    with ThreadPoolExecutor(max_workers=threads) as pool:
        scanned = list(pool.map(_scan_series, items))
    index = {series.uid: series for series in scanned}
    slices = sum(len(s.names) for s in scanned)
    log(f"index: {len(index)} series, {slices} slice headers in {time.time() - started:.0f}s")
    return index


# ------------------------------------------------------- ordering dialects --
def _natural_key(name):
    return tuple(int(x) if x.isdigit() else x.lower() for x in re.split(r"(\d+)", str(name)))


def _sorted_view(series):
    """Index arrays reordered to match sorted_names."""
    position = {name: i for i, name in enumerate(series.names)}
    take = np.array([position[name] for name in series.sorted_names], np.int64)
    return take


def _order_normal(series):
    """Branch A: signed distance along the slice normal; any gap keeps file order."""
    take = _sorted_view(series)
    ipp, iop, inst = series.ipp[take], series.iop[take], series.inst[take]
    keys = []
    for i in range(len(take)):
        if np.isfinite(ipp[i]).all() and np.isfinite(iop[i]).all():
            keys.append(float(np.dot(ipp[i], np.cross(iop[i, :3], iop[i, 3:]))))
        elif np.isfinite(inst[i]):
            keys.append(float(inst[i]))
        else:
            return list(series.sorted_names)
    return [name for _, name in sorted(zip(keys, series.sorted_names), key=lambda t: t[0])]


def _order_dominant_axis(series):
    """Branch C (legacy rules): sort along the axis of greatest travel."""
    take = _sorted_view(series)
    ipp, inst = series.ipp[take], series.inst[take]
    names = series.sorted_names
    placed = np.isfinite(ipp).all(1)
    need = max(2, int(0.8 * len(names)))
    rows = list(range(len(names)))
    if placed.sum() >= need:
        axis = int(np.argmax(np.ptp(ipp[placed], axis=0)))
        spare = float(np.nanmedian(ipp[placed][:, axis]))
        rows.sort(key=lambda i: (
            float(ipp[i, axis]) if placed[i] else spare,
            inst[i] if np.isfinite(inst[i]) else float("inf"),
            i,
        ))
    elif np.isfinite(inst).sum() >= need:
        rows.sort(key=lambda i: (inst[i] if np.isfinite(inst[i]) else float("inf"), i))
    else:
        rows.sort(key=lambda i: _natural_key(names[i]))
    return [names[i] for i in rows]


def _order_instance(series, cap=256):
    """Branch B: InstanceNumber order over the first `cap` readable slices."""
    keyed = []
    for i, name in enumerate(series.names):
        if np.isfinite(series.inst[i]):
            keyed.append((int(series.inst[i]), name))
        if len(keyed) >= cap:
            break
    return [name for _, name in sorted(keyed)]


def _order_position(series):
    """Branch D: normal-distance order, falling back to InstanceNumber then zero."""
    keys = []
    for i in range(len(series.names)):
        if np.isfinite(series.ipp[i]).all() and np.isfinite(series.iop[i]).all():
            keys.append(float(np.dot(series.ipp[i], np.cross(series.iop[i, :3], series.iop[i, 3:]))))
        elif np.isfinite(series.inst[i]):
            keys.append(float(series.inst[i]))
        else:
            keys.append(0.0)
    order = sorted(range(len(keys)), key=lambda i: keys[i])
    return [series.names[i] for i in order]


_ORDER_RULES = {
    "normal": _order_normal,
    "dominant_axis": _order_dominant_axis,
    "instance": _order_instance,
    "position": _order_position,
}


def raptor_files(series):
    """Branch D wants (name, pixel spacing) pairs plus the series median spacing."""
    names = series.order("position")
    position = {name: i for i, name in enumerate(series.names)}
    spacing = [
        float(series.spacing[position[name]]) if np.isfinite(series.spacing[position[name]]) else 0.5
        for name in names
    ]
    observed = series.spacing[np.isfinite(series.spacing)]
    median = float(np.median(observed)) if len(observed) else 0.5
    return list(zip(names, spacing)), median


# ------------------------------------------------------ series-level frame --
_SEP = re.compile(r"[_\-.]")
_FATSAT_RX = re.compile(
    r"\bfs\b|fatsat|fat sat|\bstir\b|\bspair\b|\bspir\b|\bwe\b|water excit|\btirm\b|\bsting\b|\bfatsup\b"
)
_T1_RX = re.compile(r"\bt1\b|\bt1w\b")
_T2_RX = re.compile(r"\bt2\b|\bt2w\b")
_PD_RX = re.compile(r"\bpd\b|\bpdw\b|proton|\bdp\b|dens")
FATSAT_OPTS = {"FS", "FATSAT", "FAT_SAT", "FSAT"}


def index_frame(index):
    rows = []
    for series in index.values():
        row = {
            "StudyInstanceUID": series.study,
            "SeriesInstanceUID": series.uid,
            "n_slices": len(series.names),
        }
        row.update(series.tags)
        rows.append(row)
    frame = pd.DataFrame(rows)
    for tag in SERIES_TAGS:
        if tag not in frame.columns:
            frame[tag] = None
    return frame


def annotate(frame):
    """Contrast weighting and fat-saturation flags — unchanged rules."""
    frame = frame.copy()
    desc = frame["SeriesDescription"].fillna("") + " " + frame["SequenceName"].fillna("")
    desc = desc.str.lower().str.replace(_SEP, " ", regex=True)
    options = frame["ScanOptions"].fillna("").str.upper().str.split("|")
    frame["fatsat"] = desc.str.contains(_FATSAT_RX) | options.apply(
        lambda tokens: any(token.strip() in FATSAT_OPTS for token in tokens)
    )
    tr = pd.to_numeric(frame["RepetitionTime"], errors="coerce")
    te = pd.to_numeric(frame["EchoTime"], errors="coerce")
    gre = frame["ScanningSequence"].fillna("").str.upper().str.contains("GR")
    t1, t2, pdw = desc.str.contains(_T1_RX), desc.str.contains(_T2_RX), desc.str.contains(_PD_RX)
    frame["weight"] = np.where(
        t1 & ~t2 & ~pdw, "T1",
        np.where(t2 & ~pdw, "T2",
                 np.where(pdw, "PD",
                          np.where(gre, "GRE",
                                   np.where(tr < 800, "T1",
                                            np.where(te > 60, "T2",
                                                     np.where(tr >= 800, "PD", "UNK")))))),
    )
    frame["fluid"] = np.isin(frame["weight"], ["PD", "T2"])
    frame["px"] = pd.to_numeric(
        frame["PixelSpacing"].fillna("").str.split("|").str[0].replace("", np.nan), errors="coerce"
    )
    return frame


# ------------------------------------------------------------- laterality --
def _hdr_vec(value, n):
    if not isinstance(value, str):
        return None
    try:
        parts = [float(x) for x in value.split("|")]
    except ValueError:
        return None
    return np.array(parts) if len(parts) >= n else None


def side_from_geometry(frame):
    centres = {}
    for row in frame.itertuples(index=False):
        ipp = _hdr_vec(getattr(row, "ImagePositionPatient", None), 3)
        iop = _hdr_vec(getattr(row, "ImageOrientationPatient", None), 6)
        spacing = _hdr_vec(getattr(row, "PixelSpacing", None), 2)
        rows_, cols = getattr(row, "Rows", None), getattr(row, "Columns", None)
        if ipp is None or iop is None or spacing is None or not rows_ or not cols:
            continue
        try:
            centre = (ipp[:3]
                      + iop[:3] * spacing[1] * float(cols) / 2
                      + iop[3:6] * spacing[0] * float(rows_) / 2)
        except (TypeError, ValueError):
            continue
        centres.setdefault(row.StudyInstanceUID, []).append(float(centre[0]))
    return {
        study: (None if abs(float(np.median(xs))) < LAT_MIN_OFFSET_MM
                else "R" if float(np.median(xs)) < 0 else "L")
        for study, xs in centres.items()
    }


def side_from_corner_x(frame):
    out = {}
    for study, group in frame.groupby("StudyInstanceUID"):
        xs = [
            float(v[0]) for v in
            (_hdr_vec(getattr(row, "ImagePositionPatient", None), 3)
             for row in group.itertuples(index=False))
            if v is not None and np.isfinite(v).all()
        ]
        if not xs:
            out[study] = None
            continue
        x = float(np.median(xs))
        out[study] = None if abs(x) < LEGACY_LAT_OFFSET_MM else "R" if x < 0 else "L"
    return out


def lat_of(frame, rules, tag=""):
    geometry = side_from_corner_x(frame) if rules["lat"] == "corner_x" else side_from_geometry(frame)
    sides, n_tag, n_geo, n_none, n_disagree = {}, 0, 0, 0, 0
    for study, group in frame.groupby("StudyInstanceUID"):
        values = [str(x).strip().upper() for x in group["Laterality"].dropna()]
        if rules["lat"] == "corner_x" and "ImageLaterality" in group.columns:
            values += [str(x).strip().upper() for x in group["ImageLaterality"].dropna()]
        values = [v[0] for v in values if v and v[0] in ("L", "R")]
        side = values[0] if values else None
        if side is not None:
            n_tag += 1
            n_disagree += geometry.get(study) is not None and geometry[study] != side
        else:
            side = geometry.get(study)
            n_geo += side is not None
            n_none += side is None
        sides[study] = side
    log(f"{tag}laterality: {n_tag} tagged, {n_geo} from geometry, {n_none} unresolved, "
        f"{n_disagree} disagreements")
    return sides


# ------------------------------------------------------------ pixel decode --
def decode_slice(series, name, mode):
    """Decode one slice. `mode` picks the dialect a branch was trained under:

    rescale : pixel_array * RescaleSlope + RescaleIntercept   (branches A, C)
    raw     : pixel_array as stored                            (branch B)
    lut     : modality LUT + MONOCHROME1 inversion             (branch D)

    Callers that need the same slice twice (branch C's three specs, branch D's
    two spans) memoize it themselves for the length of one series, which keeps
    peak memory at one series rather than one dataset.
    """
    try:
        dataset = pydicom.dcmread(series.path(name), force=True)
        if mode == "lut":
            from pydicom.pixel_data_handlers.util import apply_modality_lut

            array = apply_modality_lut(dataset.pixel_array, dataset).astype(np.float32)
            if str(getattr(dataset, "PhotometricInterpretation", "")) == "MONOCHROME1":
                array = array.max() - array
            return array
        array = dataset.pixel_array.astype(np.float32)
        if mode == "rescale":
            slope = float(getattr(dataset, "RescaleSlope", 1) or 1)
            intercept = float(getattr(dataset, "RescaleIntercept", 0) or 0)
            array = array * slope + intercept
        return array
    except Exception:
        return None


INDEX = build_index()
SERIES_FRAME = annotate(index_frame(INDEX))
log(f"series frame: {SERIES_FRAME.shape}, {SERIES_FRAME.fatsat.sum()} fat-saturated")

## 3. Fixed-size inputs from studies that are all shaped differently

No two studies agree on how many series they hold or how many slices those series
carry, and anything with a fixed input has to be handed one. The answer used
throughout is *slots*: a fixed, ordered set of (plane, contrast, fat-saturation)
positions, filled where the study has a matching series and left as zeros where it
does not.

Cropping is in millimetres using the DICOM pixel spacing rather than in pixels, so a
knee occupies the same fraction of the frame whether the scan is 0.3 or 0.5 mm per
pixel. `FLAGS["coverage_repair"]` is the lever against the tie-block problem
described at the top.

In [ ]:
# ============================================================================
# 3. Volume construction
# ----------------------------------------------------------------------------
# `build_caches` takes a *list* of PixelSpecs and fills them in one sweep: a
# slice that several specs need is decoded once. Branch C used to run three
# full sweeps over the same fat-saturated series; it now runs one.
# ============================================================================
def pick_slots(frame, spec, plane_of=None):
    """Choose one series per slot per study. Unchanged selection rules."""
    plane_of = PLANE_OF if plane_of is None else plane_of
    frame = frame.copy()
    frame["plane"] = frame["SeriesInstanceUID"].map(plane_of)
    rules = spec.rule
    chosen = {}
    for study, group in frame.groupby("StudyInstanceUID"):
        picked = {}
        for name, plane, fluid, fatsat in spec.slots:
            selector = (group["plane"] == plane) & (group["fatsat"] == fatsat)
            if fluid is not None:
                selector &= group["fluid"] == fluid
            candidates = group[selector]
            if len(candidates) == 0 and rules["slot_fallback"] and fluid is False:
                candidates = group[(group["plane"] == plane) & ~group["fatsat"]]
            if len(candidates):
                picked[name] = candidates.sort_values("n_slices", ascending=False).iloc[0]
        if FLAGS["coverage_repair"] and len(picked) < len(spec.slots):
            picked = _repair_coverage(picked, group, spec)
        chosen[study] = picked
    return chosen


def _repair_coverage(picked, group, spec):
    """A slot left empty is fed to the model as zeros, and a study whose slots
    are all empty falls back to a constant 0.5 — under ROC-AUC a tie block is
    dead weight. When enabled, an empty slot borrows the largest unused series
    of the same plane before giving up on it."""
    used = {record.SeriesInstanceUID for record in picked.values()}
    for name, plane, _, _ in spec.slots:
        if name in picked:
            continue
        spare = group[(group["plane"] == plane) & (~group["SeriesInstanceUID"].isin(used))]
        if len(spare):
            record = spare.sort_values("n_slices", ascending=False).iloc[0]
            picked[name] = record
            used.add(record.SeriesInstanceUID)
    return picked


def slice_indices(n_files, spec):
    """Evenly spaced positions inside the configured band of the stack."""
    if n_files == 0:
        return np.zeros(0, np.int64)
    low, high = int(spec.band[0] * (n_files - 1)), int(spec.band[1] * (n_files - 1))
    idx = (np.unique(np.linspace(low, high, spec.slices).astype(int))
           if high > low else np.array([n_files // 2]))
    while len(idx) < spec.slices:
        idx = np.append(idx, idx[-1])
    return idx[: spec.slices]


def slot_image(planes, pixel_mm, spec):
    """Percentile-window, millimetre-crop and resize one slot into uint8."""
    out_size = spec.img
    got = [i for i, plane in enumerate(planes) if plane is not None]
    if spec.rule["decode_fill"] == "zero":
        planes = [np.zeros((out_size, out_size), np.float32) if p is None else p for p in planes]
        got = list(range(len(planes)))
    if not got:
        return None
    if len(got) < len(planes):
        for i, plane in enumerate(planes):
            if plane is None:
                planes[i] = planes[min(got, key=lambda j: abs(j - i))]
    shape = planes[0].shape
    planes = [p if p.shape == shape else np.zeros(shape, np.float32) for p in planes]
    volume = np.stack(planes)
    if pixel_mm and np.isfinite(pixel_mm) and pixel_mm > 0:
        want = int(round(spec.crop_mm / pixel_mm))
        height, width = shape
        if 16 < want < min(height, width):
            cy, cx = height // 2, width // 2
            half = want // 2
            volume = volume[:, max(0, cy - half):cy + half, max(0, cx - half):cx + half]
    low, high = np.percentile(volume, [1, 99])
    volume = np.clip((volume - low) / max(high - low, 1e-06), 0, 1)
    tensor = torch.from_numpy(np.ascontiguousarray(volume)).unsqueeze(0)
    tensor = F.interpolate(tensor, size=(out_size, out_size), mode="bilinear", align_corners=False)
    return (tensor.squeeze(0) * 255).round().clamp(0, 255).to(torch.uint8).numpy()


def normalise_laterality(image, plane, side):
    if side != "R":
        return image
    return image[..., ::-1].copy() if plane in ("Coronal", "Axial") else image[::-1].copy()


def build_caches(specs, frame, tag, mode="rescale", threads=PIX_THREADS):
    """Fill every spec in `specs` from a single pass over the DICOM store.

    Returns {spec.name: (studies, cache uint8[N, slot, slice, img, img], mask)}.
    """
    specs = list(specs)
    studies = sorted(frame["StudyInstanceUID"].unique())
    position = {study: i for i, study in enumerate(studies)}
    slot_maps = {spec.name: pick_slots(frame, spec) for spec in specs}
    lat_maps = {}
    for rules in {spec.rules for spec in specs}:
        lat_maps[rules] = lat_of(frame, RULES_NATIVE if rules == "native" else RULES_LEGACY,
                                 f"{tag}/{rules} ")

    caches, masks = {}, {}
    total = 0
    for spec in specs:
        caches[spec.name] = np.zeros(
            (len(studies), spec.n_slot, spec.slices, spec.img, spec.img), np.uint8
        )
        masks[spec.name] = np.zeros((len(studies), spec.n_slot), np.float32)
        total += caches[spec.name].nbytes
    log(f"{tag}: {len(studies)} studies x {len(specs)} spec(s) = {total / 1024 ** 3:.1f} GB "
        f"({available_gb():.0f} GB available)")

    # One job per (study, series): every spec that wants this series is served
    # from the same decode.
    jobs = {}
    for spec in specs:
        for study, picked in slot_maps[spec.name].items():
            for slot_index, (slot_name, plane, _, _) in enumerate(spec.slots):
                record = picked.get(slot_name)
                if record is None:
                    continue
                jobs.setdefault((study, record.SeriesInstanceUID), []).append(
                    (spec, slot_index, plane, float(record.px) if pd.notna(record.px) else np.nan)
                )
    log(f"{tag}: {len(jobs)} series to decode for {sum(len(v) for v in jobs.values())} slot fills")

    def run(item):
        (study, series_uid), wants = item
        series = INDEX.get(series_uid)
        if series is None or not series.names:
            return []
        wanted_names, per_spec = {}, []
        for spec, slot_index, plane, pixel_mm in wants:
            order = series.order(spec.rule["order"])
            if not order:
                continue
            names = [order[int(i)] for i in slice_indices(len(order), spec)]
            per_spec.append((spec, slot_index, plane, pixel_mm, names))
            for name in names:
                wanted_names[name] = None
        decoded = {name: decode_slice(series, name, mode) for name in wanted_names}
        results = []
        for spec, slot_index, plane, pixel_mm, names in per_spec:
            image = slot_image([decoded[name] for name in names], pixel_mm, spec)
            if image is None:
                continue
            side = lat_maps[spec.rules].get(study)
            results.append((spec.name, study, slot_index, normalise_laterality(image, plane, side)))
        return results

    done, started = 0, time.time()
    items = list(jobs.items())
    with ThreadPoolExecutor(max_workers=threads) as pool:
        for results in pool.map(run, items):
            for spec_name, study, slot_index, image in results:
                caches[spec_name][position[study], slot_index] = image
                masks[spec_name][position[study], slot_index] = 1.0
            done += 1
            if done % 2048 == 0:
                log(f"  {tag} {done}/{len(items)}")
            if time.time() - T0 > TIME_BUDGET:
                log(f"  {tag}: time budget reached during decode")
                break
    for spec in specs:
        log(f"{tag}/{spec.name}: {int(masks[spec.name].sum())} slots filled "
            f"in {time.time() - started:.0f}s")
    gc.collect()
    return {spec.name: (studies, caches[spec.name], masks[spec.name]) for spec in specs}


# --------------------------------------------- branch B: 16-slice volumes ---
FOLD_SLOTS = (("Sagittal", 1), ("Sagittal", 0), ("Coronal", 1), ("Coronal", 0),
              ("Axial", 1), ("Axial", 0))
FOLD_SIZE, FOLD_SLICES, FOLD_CROP_MM, FOLD_BAND = 336, 16, 130.0, (0.12, 0.88)


def fold_plan(study):
    """Resolve slot -> ordered file paths in the parent process (no header I/O
    in the workers, which is what the original paid for on every study)."""
    rows = SERIES_ROWS.get(study, [])
    frame = pd.DataFrame(rows)
    plan = []
    if not len(frame):
        return plan
    for slot_index, (plane, fatsat) in enumerate(FOLD_SLOTS):
        match = frame[(frame.Anatomical_Plane == plane) & (frame.Fat_Suppression == fatsat)]
        if match.empty:
            continue
        series = INDEX.get(str(match.iloc[0].SeriesInstanceUID))
        if series is None:
            continue
        names = series.order("instance")
        if not names:
            continue
        first = series.names.index(names[0])
        flip = plane != "Sagittal" and float(
            series.ipp[first, 0] if np.isfinite(series.ipp[first, 0]) else 0.0
        ) < 0
        low = int(round(FOLD_BAND[0] * (len(names) - 1)))
        high = int(round(FOLD_BAND[1] * (len(names) - 1)))
        available = list(range(low, high + 1))
        if len(available) >= FOLD_SLICES:
            picks = [available[int(round(t))]
                     for t in np.linspace(0, len(available) - 1, FOLD_SLICES)]
            offset = 0
        else:
            picks, offset = available, (FOLD_SLICES - len(available)) // 2
        paths = [series.path(name) for name in names]
        plan.append((slot_index, paths, bool(flip), [int(p) for p in picks], int(offset)))
    return plan


def _fold_render(path, flip):
    import cv2 as _cv2
    import numpy as _np
    import pydicom as _pydicom

    try:
        dataset = _pydicom.dcmread(path)
        array = dataset.pixel_array.astype(_np.float32)
    except Exception:
        return None
    try:
        spacing = float(dataset.PixelSpacing[0])
    except Exception:
        spacing = FOLD_CROP_MM / max(array.shape)
    half = int(round(FOLD_CROP_MM / spacing / 2))
    cy, cx = array.shape[0] // 2, array.shape[1] // 2
    crop = array[max(0, cy - half):min(array.shape[0], cy + half),
                 max(0, cx - half):min(array.shape[1], cx + half)]
    if crop.size == 0:
        return None
    low, high = _np.percentile(crop[::4, ::4], [1, 99])
    image = _np.clip((crop - low) / max(high - low, 1e-06), 0, 1)
    image = _cv2.resize(image, (FOLD_SIZE, FOLD_SIZE), interpolation=_cv2.INTER_AREA)
    return image[:, ::-1].copy() if flip else image


def fold_build(args):
    """Worker: pixels only."""
    study_index, plan = args
    volume = np.zeros((len(FOLD_SLOTS), FOLD_SLICES, FOLD_SIZE, FOLD_SIZE), np.uint8)
    mask = np.zeros(len(FOLD_SLOTS), np.uint8)
    for slot_index, paths, flip, picks, offset in plan:
        for cursor, position in enumerate(picks):
            image = _fold_render(paths[position], flip)
            if image is None:
                image = _fold_render(paths[min(len(paths) - 1, position + 1)], flip)
            if image is not None:
                volume[slot_index, offset + cursor] = (image * 255).astype(np.uint8)
        mask[slot_index] = len(picks)
    return study_index, volume, mask


# ------------------------------------------ branch D: 64-slice dual volumes --
RAPTOR_IMG, RAPTOR_CROP_MM = 336, 140.0
RAPTOR_SLOTS = (("Sagittal", 1, 18), ("Sagittal", 0, 14), ("Coronal", 1, 12),
                ("Coronal", 0, 8), ("Axial", -1, 12))
RAPTOR_MAXS = sum(slot[2] for slot in RAPTOR_SLOTS)
RAPTOR_PRIMARY_SPAN = (0.02, 0.98)
RAPTOR_LEGACY_SPAN = (0.06, 0.94)


def _raptor_pick(rows, plane, fluid, used):
    candidates = [r for r in rows
                  if r["Anatomical_Plane"] == plane and r["SeriesInstanceUID"] not in used]
    if fluid in (0, 1):
        preferred = [r for r in candidates if int(r.get("Fluid_Sensitive", 0) or 0) == fluid]
        if preferred:
            return preferred[0]
    return candidates[0] if candidates else None


def _raptor_crop_resize(array, spacing):
    height, width = array.shape
    size = min(int(round(RAPTOR_CROP_MM / max(spacing, 1e-3))), min(height, width))
    y0, x0 = (height - size) // 2, (width - size) // 2
    array = array[y0:y0 + size, x0:x0 + size]
    return cv2.resize(array, (RAPTOR_IMG, RAPTOR_IMG), interpolation=cv2.INTER_AREA)


def _raptor_fill(volume, offset, picks, decoded, files, median_ps):
    arrays = [decoded.get(min(int(p), len(files) - 1)) for p in picks]
    spacings = [files[min(int(p), len(files) - 1)][1] or median_ps for p in picks]
    valid = [a for a in arrays if a is not None]
    if valid:
        low, high = np.percentile(np.concatenate([a.ravel() for a in valid]), [2.0, 98.0])
    else:
        low, high = 0.0, 1.0
    for cursor, (array, spacing) in enumerate(zip(arrays, spacings)):
        index = offset + cursor
        if index >= RAPTOR_MAXS:
            break
        if array is None:
            continue
        normalised = np.clip((array - low) / (high - low + 1e-6), 0, 1)
        volume[index] = (_raptor_crop_resize(normalised, spacing) * 255).astype(np.uint8)


def raptor_volumes(study):
    """Exact MaxSpan and legacy-span volumes, reading each DICOM once."""
    rows = SERIES_ROWS.get(study, [])
    primary = np.zeros((RAPTOR_MAXS, RAPTOR_IMG, RAPTOR_IMG), np.uint8)
    legacy = np.zeros_like(primary)
    used, offset = set(), 0
    for plane, fluid, count in RAPTOR_SLOTS:
        record = _raptor_pick(rows, plane, fluid, used)
        if record is None:
            offset += count
            continue
        used.add(record["SeriesInstanceUID"])
        series = INDEX.get(str(record["SeriesInstanceUID"]))
        if series is None or not series.names:
            offset += count
            continue
        files, median_ps = raptor_files(series)
        number = len(files)
        spans = []
        for span_lo, span_hi in (RAPTOR_PRIMARY_SPAN, RAPTOR_LEGACY_SPAN):
            low = int(number * span_lo)
            high = max(int(number * span_hi) - 1, low)
            spans.append(np.linspace(low, high, count).round().astype(int)
                         if number > 1 else np.zeros(count, dtype=int))
        decoded = {}
        for position in sorted({int(p) for span in spans for p in span}):
            position = min(position, number - 1)
            decoded[position] = decode_slice(series, files[position][0], "lut")
        _raptor_fill(primary, offset, spans[0], decoded, files, median_ps)
        _raptor_fill(legacy, offset, spans[1], decoded, files, median_ps)
        offset += count
        if offset >= RAPTOR_MAXS:
            break
    primary_mask = (primary.reshape(RAPTOR_MAXS, -1).sum(1) > 0).astype(np.uint8)
    legacy_mask = (legacy.reshape(RAPTOR_MAXS, -1).sum(1) > 0).astype(np.uint8)
    return primary, primary_mask, legacy, legacy_mask

## 4. Branch A: per-finding attention over slots

A cruciate tear may be visible in two sagittal slices while osteoarthritis is spread
across many coronal ones. A single pooled score forces both to share one notion of
which slot matters; giving each finding its own attention weights does not.
`SLOT_PRIOR_TABLE` seeds that attention with the slots a radiologist would actually
look at for each finding.

The fingerprint check replays a fixed random input through the loaded weights and
compares it against a value stored at fit time. It catches a preprocessing drift that
would otherwise load cleanly and quietly cost accuracy.

In [ ]:
# ============================================================================
# 4. Branch A model — DINOv2 backbone with a per-target slot attention head
# ============================================================================
POOL_PARTS = {"cls_mean": 2, "cls_mean_focal": 3}
SLOT_PRIOR_TABLE = {
    "ACL": (0, 3, 5), "MCL": (1, 4), "Medial Meniscus": (0, 1, 3, 4),
    "Lateral Meniscus": (0, 1, 3, 4), "Medial OA": (1, 4, 5), "Lateral OA": (1, 4, 5),
    "PF OA": (0, 2, 5), "Effusion": (0, 2), "Synovitis": (0, 2), "Baker's": (0,),
    "Contusion": (0, 1, 2), "Fracture": (0, 1, 2, 4, 5),
}
SLOT_PRIOR_STRENGTH = 0.55
FINGERPRINT_TOL = 0.002


class SlotHead(nn.Module):
    def __init__(self, dim, n_slot, n_out, hidden=256, p=0.2, prior=False):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop = nn.Dropout(p)
        self.out = nn.Linear(hidden, n_out)
        self.hidden = hidden
        self.prior = prior
        if prior:
            table = torch.zeros(n_out, n_slot)
            if n_slot == len(SLOTS_RECOVERED) and n_out == len(TARGETS):
                for target, slots in SLOT_PRIOR_TABLE.items():
                    if target in TARGETS:
                        table[TARGETS.index(target), list(slots)] = SLOT_PRIOR_STRENGTH
            self.register_buffer("slot_prior", table)

    def forward(self, x, mask):
        h = self.proj(x) + self.slot_emb
        attention = torch.einsum("bsh,oh->bos", h, self.query) / self.hidden ** 0.5
        if self.prior:
            attention = attention + self.slot_prior.unsqueeze(0)
        attention = attention.masked_fill(mask.unsqueeze(1) < 0.5, -10000.0).softmax(-1)
        context = self.drop(torch.einsum("bos,bsh->boh", attention, h))
        return (context * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias


class DinoSlotModel(nn.Module):
    def __init__(self, backbone, dim, n_slot, pool="cls_mean", prior=False):
        super().__init__()
        self.backbone = backbone
        self.pool = pool
        self.head = SlotHead(dim * POOL_PARTS[pool], n_slot, len(TARGETS), prior=prior)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, images, mask, img_size=None):
        batch, slots = images.shape[:2]
        x = images.reshape(batch * slots, *images.shape[2:]).float().div_(255.0)
        if img_size is not None and img_size != x.shape[-1]:
            x = F.interpolate(x, size=(img_size, img_size), mode="bilinear", align_corners=False)
        x = (x - self.mean) / self.std
        out = self.backbone(pixel_values=x).last_hidden_state
        patch = out[:, 1:]
        parts = [out[:, 0], patch.mean(1)]
        if self.pool == "cls_mean_focal":
            k = max(1, patch.shape[1] // 8)
            parts.append(patch.topk(k, dim=1).values.mean(1))
        features = torch.cat(parts, dim=1).reshape(batch, slots, -1)
        return self.head(features, mask)


def build_dino_model(unfreeze_last, n_slot, variant="small", pool="cls_mean", prior=False):
    from transformers import AutoModel

    if not (DINO_DIR / "config.json").is_file():
        raise FileNotFoundError(DINO_DIR)
    backbone = AutoModel.from_pretrained(str(DINO_DIR))
    layers = len(backbone.encoder.layer)
    for parameter in backbone.parameters():
        parameter.requires_grad = False
    for block in backbone.encoder.layer[max(0, layers - unfreeze_last):]:
        for parameter in block.parameters():
            parameter.requires_grad = True
    for parameter in backbone.layernorm.parameters():
        parameter.requires_grad = True
    return DinoSlotModel(backbone, backbone.config.hidden_size, n_slot, pool=pool, prior=prior)


def fingerprint(model, device, img_size, n_slot, group, seed=SEED):
    """Replay the stored forward signature so a preprocessing drift is caught
    before it silently costs accuracy."""
    generator = torch.Generator().manual_seed(seed)
    images = torch.randint(0, 256, (2, n_slot, group, img_size, img_size),
                           generator=generator, dtype=torch.uint8).to(device)
    mask = torch.ones(2, n_slot, device=device)
    mask[1, -1] = 0.0
    was_training = model.training
    model.eval()
    with torch.no_grad():
        out = model(images, mask, img_size).float().cpu().numpy()
    if was_training:
        model.train()
    return out


def check_fingerprint(model, device, spec, expected, tag=""):
    got = fingerprint(model, device, spec.img, spec.n_slot, spec.group)
    expected = np.asarray(expected, np.float32)
    if got.shape != expected.shape:
        raise WeightsError(f"{tag}fingerprint shape {got.shape} != stored {expected.shape}")
    delta = float(np.abs(got - expected).max())
    if delta > FINGERPRINT_TOL:
        raise WeightsError(
            f"{tag}fingerprint differs by {delta:.4g} (tolerance {FINGERPRINT_TOL:g}); the weights "
            "load but no longer compute what they computed when fitted"
        )
    log(f"  {tag}fingerprint matches within {delta:.2g}")
    return delta

## 5. Branch B: fold ensemble

These definitions are load-bearing — state-dict keys and shapes must match the
checkpoints — so the module layout is kept as fitted. The interesting part is the
pooling family: mean-max, label attention, token cross-attention and two residual
variants, selected per checkpoint by its own config.

In [ ]:
# ============================================================================
# 5. Branch B models — timm encoders with segment pooling over present slots
# ----------------------------------------------------------------------------
# Definitions are load-bearing: state_dict keys and shapes must match the
# checkpoints, so the module layout is kept as fitted.
# ============================================================================
import timm

N_SLOT_TYPES, MASK_IDX = 6, 0
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
N_PLANE, N_CONTRAST = 3, 2
_PLANE_OF = lambda slot: torch.clamp(slot - 1, 0, 5) // 2
_CONTRAST_OF = lambda slot: torch.clamp(slot - 1, 0, 5) % 2


def segment_softmax(scores, segment, batch):
    _, k = scores.shape
    index = segment.unsqueeze(1).expand(-1, k)
    peak = torch.full((batch, k), float("-inf"), device=scores.device, dtype=scores.dtype)
    peak = peak.scatter_reduce(0, index, scores, reduce="amax", include_self=True)
    exponent = (scores - peak[segment]).exp()
    total = torch.zeros(batch, k, device=scores.device, dtype=scores.dtype).index_add_(
        0, segment, exponent
    )
    return exponent / total[segment].clamp(min=1e-06)


def _seg_mean_max(values, segment, batch):
    dim = values.shape[1]
    count = torch.zeros(batch, device=values.device, dtype=values.dtype).index_add_(
        0, segment, torch.ones(values.shape[0], device=values.device, dtype=values.dtype)
    )
    mean = torch.zeros(batch, dim, device=values.device, dtype=values.dtype).index_add_(
        0, segment, values
    ) / count.clamp(min=1).unsqueeze(1)
    peak = torch.full((batch, dim), -10000.0, device=values.device, dtype=values.dtype)
    peak = peak.scatter_reduce(0, segment.unsqueeze(1).expand(-1, dim), values,
                               reduce="amax", include_self=True)
    return torch.cat([mean, peak], 1)


def _pad_kv(x, segment, batch, norm):
    length, patches, dim = x.shape
    count = torch.bincount(segment, minlength=batch)
    span = int(count.max().item())
    starts = torch.cumsum(count, 0) - count
    position = torch.arange(length, device=x.device) - starts[segment]
    padded = x.new_zeros(batch, span, patches, dim)
    padded[segment, position] = x
    keep = torch.zeros(batch, span, dtype=torch.bool, device=x.device)
    keep[segment, position] = True
    return norm(padded.reshape(batch, span * patches, dim)), ~keep.repeat_interleave(patches, dim=1)


class MeanMaxPool(nn.Module):
    def forward(self, features, segment, batch, slot=None, return_attn=False):
        return _seg_mean_max(features, segment, batch), None


class LabelAttentionPool(nn.Module):
    def __init__(self, dim, n_labels=12, n_heads=4, slot_bias=True):
        super().__init__()
        self.d, self.k, self.h = dim, n_labels, n_heads
        self.q = nn.Parameter(torch.randn(n_labels, dim) * 0.02)
        self.key, self.val = nn.Linear(dim, dim), nn.Linear(dim, dim)
        self.slot_bias = nn.Parameter(torch.zeros(n_labels, N_SLOT_TYPES + 1)) if slot_bias else None

    def forward(self, features, segment, batch, slot=None, return_attn=False):
        scores = self.key(features) @ self.q.t() / self.d ** 0.5
        if self.slot_bias is not None and slot is not None:
            scores = scores + self.slot_bias.t()[slot]
        attention = segment_softmax(scores, segment, batch)
        out = torch.zeros(batch, self.k, self.d, device=features.device, dtype=features.dtype)
        out = out.index_add_(0, segment, attention.unsqueeze(-1) * self.val(features).unsqueeze(1))
        return out, attention


class TokenXAttnPool(nn.Module):
    def __init__(self, dim, n_labels=12, n_heads=6, dropout=0.2):
        super().__init__()
        self.d, self.k = dim, n_labels
        self.q = nn.Parameter(torch.randn(n_labels, dim) * 0.02)
        self.slot_emb = nn.Embedding(N_SLOT_TYPES + 1, dim, padding_idx=0)
        self.kv_norm = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, n_heads, dropout=dropout, batch_first=True)

    def forward(self, tokens, segment, batch, slot=None, return_attn=False):
        kv = tokens + self.slot_emb(slot).unsqueeze(1)
        padded, key_padding = _pad_kv(kv, segment, batch, self.kv_norm)
        query = self.q.unsqueeze(0).expand(batch, -1, -1)
        attended, weights = self.attn(query, padded, padded, key_padding_mask=key_padding,
                                      need_weights=return_attn, average_attn_weights=True)
        cls = tokens[:, 0]
        base = _seg_mean_max(cls, segment, batch).unsqueeze(1).expand(-1, self.k, -1)
        return torch.cat([attended, base], -1), weights


class ViTSlotToken(nn.Module):
    """Injects a learned slot token next to the ViT prefix tokens."""

    def __init__(self, vit, n_cat, dim=None):
        super().__init__()
        self.vit = vit
        dim = dim or vit.embed_dim
        self.tok = nn.Embedding(n_cat + 1, dim, padding_idx=MASK_IDX)
        self.num_features = vit.num_features
        self._orig_prefix = getattr(vit, "num_prefix_tokens", 1)
        vit.num_prefix_tokens = self._orig_prefix + 1
        for block in vit.blocks:
            attention = getattr(block, "attn", None)
            if attention is not None and hasattr(attention, "num_prefix_tokens"):
                attention.num_prefix_tokens = attention.num_prefix_tokens + 1

    @staticmethod
    def _maybe(module, x):
        return x if module is None else module(x)

    def forward_features(self, x, cat):
        vit = self.vit
        x = vit.patch_embed(x)
        embedded = vit._pos_embed(x)
        rope = None
        if isinstance(embedded, tuple):
            x, rope = embedded
        else:
            x = embedded
        x = self._maybe(getattr(vit, "patch_drop", None), x)
        x = self._maybe(getattr(vit, "norm_pre", None), x)
        prefix = self._orig_prefix
        x = torch.cat([x[:, :prefix], self.tok(cat).unsqueeze(1), x[:, prefix:]], dim=1)
        if rope is not None:
            if getattr(vit, "rope_mixed", False):
                for i, block in enumerate(vit.blocks):
                    x = block(x, rope=rope[i])
            else:
                for block in vit.blocks:
                    x = block(x, rope=rope)
        else:
            x = vit.blocks(x)
        return vit.norm(x)

    def forward_head(self, x, pre_logits=True):
        return self.vit.forward_head(x, pre_logits=pre_logits)


class _GatedDepthBlock(nn.Module):
    def __init__(self, n_slice, dropout=0.0, ls_init=0.1):
        super().__init__()
        self.norm = nn.GroupNorm(1, n_slice)
        self.v = nn.Conv2d(n_slice, n_slice, 1)
        self.g = nn.Conv2d(n_slice, n_slice, 1)
        self.out = nn.Conv2d(n_slice, n_slice, 1)
        self.gamma = nn.Parameter(torch.full((n_slice, 1, 1), ls_init))
        self.drop = nn.Dropout2d(dropout) if dropout else nn.Identity()

    def forward(self, x):
        z = self.norm(x)
        return x + self.gamma * self.drop(self.out(self.v(z) * F.silu(self.g(z))))


class DepthCompress(nn.Module):
    """Collapses the slice axis to three channels before the encoder."""

    def __init__(self, n_slice=16, out_ch=3, depth=1, dropout=0.0, ls_init=0.1,
                 imagenet=True, proj_noise=0.25):
        super().__init__()
        self.imagenet = imagenet
        self.blocks = nn.ModuleList([_GatedDepthBlock(n_slice, dropout, ls_init) for _ in range(depth)])
        self.proj = nn.Conv2d(n_slice, out_ch, 1, bias=True)
        if imagenet:
            self.register_buffer("mu", torch.tensor(IMAGENET_MEAN).view(1, -1, 1, 1))
            self.register_buffer("sd", torch.tensor(IMAGENET_STD).view(1, -1, 1, 1))

    def forward(self, x):
        keep = (x.amax(dim=1, keepdim=True) > 0).to(x.dtype)
        z = x
        for block in self.blocks:
            z = block(z)
        z = self.proj(z)
        if self.imagenet:
            z = (z - self.mu.to(z.dtype)) / self.sd.to(z.dtype)
        return z * keep


class SlotDepthMixer(nn.Module):
    """Plane/contrast-conditioned smoothing along the slice axis."""

    def __init__(self, n_slice=16, ksize=5, alpha_max=0.25):
        super().__init__()
        self.n_slice, self.ksize, self.r = n_slice, ksize, ksize // 2
        self.alpha_max = alpha_max
        base = torch.tensor([1.0, 4.0, 6.0, 4.0, 1.0])
        self.register_buffer("base", base.log()[self.r:])
        units = self.r + 1
        self.shared = nn.Parameter(torch.zeros(units))
        self.plane_k = nn.Parameter(torch.zeros(N_PLANE, units))
        self.contrast_k = nn.Parameter(torch.zeros(N_CONTRAST, units))
        self.g0 = nn.Parameter(torch.zeros(()))
        self.gate_p = nn.Parameter(torch.zeros(N_PLANE))
        self.gate_c = nn.Parameter(torch.zeros(N_CONTRAST))
        index = torch.arange(n_slice)
        self.register_buffer("off", index[None, :] - index[:, None])

    def kernel(self, slot):
        half = (self.base + self.shared + self.plane_k[_PLANE_OF(slot)]
                + self.contrast_k[_CONTRAST_OF(slot)])
        full = torch.cat([half.flip(-1)[..., :self.r], half], dim=-1)
        return F.softmax(full, dim=-1)

    def alpha(self, slot):
        return self.alpha_max * torch.tanh(
            self.g0 + self.gate_p[_PLANE_OF(slot)] + self.gate_c[_CONTRAST_OF(slot)]
        )

    def forward(self, x, slot, valid):
        length, slices, height, width = x.shape
        if valid is None:
            raise ValueError("stem=mixer requires the padding mask")
        kernel = self.kernel(slot)
        weights = valid.to(kernel.dtype)
        distance = self.off + self.r
        inside = (distance >= 0) & (distance < self.ksize)
        banded = kernel[:, distance.clamp(0, self.ksize - 1)] * inside
        matrix = banded * weights[:, None, :]
        denominator = matrix.sum(-1, keepdim=True)
        eye = torch.eye(slices, device=x.device, dtype=matrix.dtype).expand(length, slices, slices)
        ok = (denominator > 1e-06) & weights[:, :, None].bool()
        matrix = torch.where(ok, matrix / denominator.clamp(min=1e-06), eye)
        alpha = self.alpha(slot)[:, None, None]
        operator = ((1.0 - alpha) * eye + alpha * matrix).to(x.dtype)
        if x.is_contiguous(memory_format=torch.channels_last) and not x.is_contiguous():
            mixed = torch.bmm(x.permute(0, 2, 3, 1).reshape(length, height * width, slices),
                              operator.transpose(1, 2))
            return mixed.reshape(length, height, width, slices).permute(0, 3, 1, 2)
        return torch.bmm(operator, x.reshape(length, slices, height * width)).reshape(
            length, slices, height, width
        )


class _GatedDelta(nn.Module):
    def __init__(self, dim, n_labels, n_heads, dropout):
        super().__init__()
        self.q = nn.Parameter(torch.randn(n_labels, dim) * 0.02)
        self.kv_norm = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, n_heads, dropout=dropout, batch_first=True)
        self.d_norm = nn.LayerNorm(dim)
        self.dw = nn.Parameter(torch.randn(n_labels, dim) * (1.0 / dim ** 0.5))
        self.db = nn.Parameter(torch.zeros(n_labels))
        self.gate = nn.Parameter(torch.zeros(n_labels))

    def delta(self, patches, segment, batch, return_attn):
        kv, key_padding = _pad_kv(patches, segment, batch, self.kv_norm)
        query = self.q.unsqueeze(0).expand(batch, -1, -1)
        attended, weights = self.attn(query, kv, kv, key_padding_mask=key_padding,
                                      need_weights=return_attn, average_attn_weights=True)
        return (self.d_norm(attended) * self.dw).sum(-1) + self.db, weights


class TokenResidualPool(_GatedDelta):
    def __init__(self, dim, n_labels=12, n_heads=6, pe=64, dropout=0.2):
        super().__init__(dim, n_labels, n_heads, dropout)
        self.base = nn.Sequential(nn.LayerNorm(2 * dim + pe), nn.Dropout(dropout),
                                  nn.Linear(2 * dim + pe, n_labels))

    def forward(self, tokens, slot, segment, batch, presence, return_attn=False):
        base = self.base(torch.cat([_seg_mean_max(tokens[:, 1:].mean(1), segment, batch), presence], 1))
        delta, weights = self.delta(tokens[:, 1:], segment, batch, return_attn)
        return base + self.gate * delta, weights


class CodexResidualPool(_GatedDelta):
    def __init__(self, dim, n_labels=12, n_heads=6, pe=64, dropout=0.2):
        super().__init__(dim, n_labels, n_heads, dropout)
        self.base = nn.Sequential(nn.LayerNorm(2 * dim + pe), nn.Dropout(dropout),
                                  nn.Linear(2 * dim + pe, n_labels))

    def forward(self, tokens, slot, segment, batch, presence, return_attn=False):
        base = self.base(torch.cat([_seg_mean_max(tokens[:, 0], segment, batch), presence], 1))
        delta, weights = self.delta(tokens[:, 1:], segment, batch, return_attn)
        return base + self.gate * delta, weights


class ClsAddPool(nn.Module):
    def __init__(self, dim, n_labels=12, pe=64, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(nn.LayerNorm(4 * dim + pe), nn.Dropout(dropout),
                                 nn.Linear(4 * dim + pe, n_labels))

    def forward(self, tokens, slot, segment, batch, presence, return_attn=False):
        return self.net(torch.cat([
            _seg_mean_max(tokens[:, 1:].mean(1), segment, batch),
            _seg_mean_max(tokens[:, 0], segment, batch),
            presence,
        ], 1)), None


class Readout(nn.Module):
    def __init__(self, pool, dim, n_labels=12, pe=64):
        super().__init__()
        self.pool_kind, self.k = pool, n_labels
        self.pres_emb = nn.Embedding(N_SLOT_TYPES + 1, pe, padding_idx=0)
        if pool in ("xres", "clsadd", "xcodex"):
            self.pool = {"xres": TokenResidualPool, "clsadd": ClsAddPool,
                         "xcodex": CodexResidualPool}[pool](dim, n_labels, pe=pe)
        elif pool in ("attn", "xattn"):
            if pool == "xattn":
                self.pool = TokenXAttnPool(dim, n_labels)
                width = 3 * dim + pe
            else:
                self.pool = LabelAttentionPool(dim, n_labels)
                width = dim + pe
            self.norm = nn.LayerNorm(width)
            self.w = nn.Parameter(torch.randn(n_labels, width) * (1.0 / width ** 0.5))
            self.b = nn.Parameter(torch.zeros(n_labels))
        else:
            self.pool = MeanMaxPool()
            self.net = nn.Sequential(nn.LayerNorm(2 * dim + pe), nn.Dropout(0.2),
                                     nn.Linear(2 * dim + pe, n_labels))
        self.drop = nn.Dropout(0.2)

    def forward(self, features, slot, segment, batch, return_attn=False):
        embedding = self.pres_emb(slot)
        presence = torch.zeros(batch, embedding.shape[1], device=features.device,
                               dtype=features.dtype).index_add_(0, segment, embedding)
        if self.pool_kind in ("xres", "clsadd", "xcodex"):
            return self.pool(features, slot, segment, batch, presence)[0]
        pooled, _ = self.pool(features, segment, batch, slot=slot, return_attn=return_attn)
        if self.pool_kind in ("attn", "xattn"):
            x = torch.cat([pooled, presence.unsqueeze(1).expand(-1, self.k, -1)], -1)
            x = self.drop(self.norm(x))
            return (x * self.w).sum(-1) + self.b
        return self.net(torch.cat([pooled, presence], 1))


class FoldNet(nn.Module):
    def __init__(self, encoder, cond, n_meta=0, pool="mean_max", stem="native", n_slice=16):
        super().__init__()
        self.enc, self.cond = encoder, cond
        self.compress = DepthCompress(n_slice, 3) if stem == "compress" else None
        self.mixer = SlotDepthMixer(n_slice) if stem == "mixer" else None
        self.tokens = pool in ("xattn", "xres", "clsadd", "xcodex")
        dim = encoder.num_features
        self.meta_mlp = nn.Sequential(nn.LayerNorm(n_meta), nn.Linear(n_meta, 128), nn.GELU(),
                                      nn.Linear(128, dim)) if n_meta > 0 else None
        self.readout = Readout(pool, dim)
        if cond == "post":
            self.slot_emb = nn.Embedding(N_SLOT_TYPES + 1, dim, padding_idx=MASK_IDX)

    def forward(self, images, slot, meta, segment, batch, valid=None):
        if self.mixer is not None:
            images = self.mixer(images, slot, valid)
        if self.compress is not None:
            images = self.compress(images)
        features = (self.enc.forward_features(images, slot) if self.cond == "token"
                    else self.enc.forward_features(images))
        if self.tokens:
            inner = getattr(self.enc, "vit", self.enc)
            prefix = getattr(self.enc, "_orig_prefix", getattr(inner, "num_prefix_tokens", 1))
            features = torch.cat([features[:, :1], features[:, prefix:]], 1)
        else:
            features = self.enc.forward_head(features, pre_logits=True)
            if features.dim() > 2:
                features = features.flatten(1)
        expand = (lambda v: v.unsqueeze(1)) if self.tokens else (lambda v: v)
        if self.cond == "post":
            features = features + expand(self.slot_emb(slot))
        if self.meta_mlp is not None and meta.shape[1] > 0:
            projected = self.meta_mlp(meta)
            features = (torch.cat([features, projected.unsqueeze(1)], 1) if self.tokens
                        else features + projected)
        return self.readout(features, slot, segment, batch)


def load_fold_models(directory=FOLD_CKPT_DIR):
    models, config = [], None
    for path in sorted(Path(directory).glob("*_f*.pt")):
        payload = torch.load(path, map_location="cpu", weights_only=False)
        config = payload["cfg"]
        stem = config.get("stem", "native")
        in_chans = 3 if stem == "compress" else config.get("n_slice", 16)
        encoder = timm.create_model(
            config["backbone"], pretrained=False, num_classes=0, in_chans=in_chans,
            **({"img_size": config["img"]} if "vit_" in config["backbone"] else {}),
        )
        if config["cond"] == "token":
            encoder = ViTSlotToken(encoder, N_SLOT_TYPES)
        model = FoldNet(encoder, config["cond"], config.get("n_meta", 0), config["pool"],
                        stem=stem, n_slice=config.get("n_slice", 16))
        missing, unexpected = model.load_state_dict(payload["state_dict"], strict=False)
        assert not missing, f"missing {missing[:5]}"
        assert not unexpected, f"unexpected {unexpected[:5]}"
        models.append(model.eval())
        log(f"  fold {payload['fold']}: {config['backbone']} pool={config['pool']}")
    if not models:
        raise WeightsError(f"no fold checkpoints under {directory}")
    assert config.get("n_meta", 0) == 0, "checkpoints expect study metadata features"
    return models, config

## 6. Branches C and D

RadImageNet gives a ResNet-50 pretrained on radiology rather than ImageNet, used
frozen as a feature extractor with five trained attention heads on top. The Raptor
arm is a CoAtNet over three-slice windows: neighbouring slices become the three
channels of one image, which buys most of the benefit of a 3D model at the cost of a
2D one.

In [ ]:
# ============================================================================
# 6. Branch C (RadImageNet ResNet-50 + attention heads) and branch D (Raptor)
# ----------------------------------------------------------------------------
# RadHead used to read its slot count from mutating globals, which made the
# head silently dependent on cell order. It now takes both explicitly.
# ============================================================================
from torchvision.models import resnet50

RAD_TOKEN_DIM, RAD_HEAD_DIM = 2048, 512
RAD_ENCODER_SHA = "08629f7e7bd3e29b8ee9522ca3f65ce4d010a7ddf74f0ea3c7e3f3d0bbab0734"
RAD_HEADS_V15_SHA = "0f465649799ecfbccaac1767844639e7ced44e1bc9babde6e4bac7c5d9b89eaa"
RAD_HEADS_E13_SHA = "ad9f19af73bfdf4e49263c0e45060dc3cb239e1195039b26dc8c0a3a6bcd1a8a"


def sha256_of(path, chunk=8 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(chunk), b""):
            digest.update(block)
    return digest.hexdigest()


def verified(path, expected_sha):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    if sha256_of(path) != expected_sha:
        raise WeightsError(f"hash mismatch for {path}")
    return path


class RadEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(*list(resnet50(weights=None).children())[:-2])

    def forward(self, image):
        return self.backbone(image).mean(dim=(2, 3))


class RadHead(nn.Module):
    def __init__(self, n_slot, n_slice):
        super().__init__()
        self.n_slot, self.n_slice = n_slot, n_slice
        self.project = nn.Sequential(nn.LayerNorm(RAD_TOKEN_DIM),
                                     nn.Linear(RAD_TOKEN_DIM, RAD_HEAD_DIM), nn.GELU())
        self.plane = nn.Parameter(torch.randn(n_slot, RAD_HEAD_DIM) * 0.01)
        self.position = nn.Parameter(torch.randn(n_slice, RAD_HEAD_DIM) * 0.01)
        self.query = nn.Parameter(torch.randn(len(TARGETS), RAD_HEAD_DIM) * 0.02)
        self.attn = nn.MultiheadAttention(RAD_HEAD_DIM, 8, dropout=0.1, batch_first=True)
        self.fuse = nn.Sequential(nn.LayerNorm(RAD_HEAD_DIM * 4),
                                  nn.Linear(RAD_HEAD_DIM * 4, RAD_HEAD_DIM), nn.GELU(),
                                  nn.Dropout(0.15))
        self.weight = nn.Parameter(torch.randn(len(TARGETS), RAD_HEAD_DIM) * 0.02)
        self.bias = nn.Parameter(torch.zeros(len(TARGETS)))

    def forward(self, feature, mask):
        token = self.project(feature.float())
        token = token.view(len(token), self.n_slot, self.n_slice, RAD_HEAD_DIM)
        token = token + self.plane[None, :, None] + self.position[None, None]
        token = token.flatten(1, 2)
        key_padding = mask <= 0
        empty = key_padding.all(1)
        if empty.any():
            key_padding = key_padding.clone()
            key_padding[empty, 0] = False
        query = self.query.unsqueeze(0).expand(len(token), -1, -1)
        attended = query + self.attn(query, token, token, key_padding_mask=key_padding,
                                     need_weights=False)[0]
        denominator = mask.sum(1, keepdim=True).clamp_min(1).unsqueeze(-1)
        mean = (token * mask.unsqueeze(-1)).sum(1, keepdim=True) / denominator
        mean = mean.expand(-1, len(TARGETS), -1)
        fused = self.fuse(torch.cat([attended, mean, torch.abs(attended - mean), attended * mean],
                                    dim=-1))
        return (fused * self.weight.unsqueeze(0)).sum(-1) + self.bias


def load_rad_heads(path, expected_sha, contract, n_slot, n_slice, device, weights_only):
    payload = torch.load(verified(path, expected_sha), map_location="cpu",
                         weights_only=weights_only)
    for key, value in contract.items():
        if payload.get(key) != value:
            raise WeightsError(f"head contract drift for {key}")
    folds = payload.get("folds")
    if not isinstance(folds, list) or len(folds) != 5:
        raise WeightsError("bundle requires exactly five heads")
    if sorted(int(record.get("fold", -1)) for record in folds) != list(range(5)):
        raise WeightsError("fold identity drift")
    heads = []
    for record in folds:
        head = RadHead(n_slot, n_slice).to(device).eval()
        head.load_state_dict(record["state_dict"], strict=True)
        heads.append(head)
    return heads


@torch.inference_mode()
def rad_encode(encoder, pixels, slot_mask, device):
    n, slots, slices, height, width = pixels.shape
    features = np.zeros((n, slots * slices, RAD_TOKEN_DIM), np.float16)
    token_mask = np.repeat(slot_mask[:, :, None], slices, axis=2).reshape(n, -1)
    valid = np.flatnonzero(token_mask.reshape(-1) > 0)
    flat = pixels.reshape(-1, height, width)
    batch = (192 if device.type == "cuda" and torch.cuda.device_count() > 1
             else 96 if device.type == "cuda" else 8)
    for start in range(0, len(valid), batch):
        indices = valid[start:start + batch]
        image = torch.from_numpy(flat[indices]).to(device).float().div_(127.5).sub_(1.0)
        image = image.unsqueeze(1).expand(-1, 3, -1, -1).contiguous()
        amp = torch.autocast("cuda") if device.type == "cuda" else contextlib.nullcontext()
        with amp:
            feature = encoder(image)
        values = feature.float().cpu().numpy()
        if not np.isfinite(values).all():
            raise WeightsError("non-finite RadImageNet feature")
        features.reshape(-1, RAD_TOKEN_DIM)[indices] = values.astype(np.float16)
    return features, token_mask.astype(np.float32)


@torch.inference_mode()
def rad_predict(head, features, masks, device, batch=64):
    out = []
    for start in range(0, len(features), batch):
        feature = torch.from_numpy(features[start:start + batch]).to(device)
        mask = torch.from_numpy(masks[start:start + batch]).to(device)
        amp = torch.autocast("cuda") if device.type == "cuda" else contextlib.nullcontext()
        with amp:
            out.append(torch.sigmoid(head(feature, mask)).float().cpu())
    return torch.cat(out).numpy()


# ------------------------------------------------------- branch D: Raptor ---
RAPTOR_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
RAPTOR_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def build_backbone(arch, pretrained=False):
    # maxvit/coatnet/convnext are conv-attention hybrids: no CLS token and no
    # interpolatable position embedding, so they must not take the ViT path
    # even though "vit" appears in the name.
    hybrid = arch.startswith(("maxvit", "maxxvit", "coatnet", "coat_", "convnext"))
    is_vit = (not hybrid) and any(k in arch for k in ("vit", "deit", "dinov2", "eva", "beit"))
    kwargs = dict(pretrained=pretrained, num_classes=0, in_chans=3)
    kwargs.update({"global_pool": "token", "dynamic_img_size": True} if is_vit
                  else {"global_pool": "avg"})
    return timm.create_model(arch, **kwargs)


class RaptorClassifier(nn.Module):
    """Per-finding attention over three-slice windows."""

    def __init__(self, backbone, dim=768, n=12, drop=0.2):
        super().__init__()
        self.backbone = backbone
        self.norm = nn.LayerNorm(dim)
        self.att = nn.Sequential(nn.Linear(dim, 256), nn.Tanh(), nn.Dropout(drop),
                                 nn.Linear(256, n))
        self.clsW = nn.Parameter(torch.zeros(n, dim))
        self.clsb = nn.Parameter(torch.zeros(n))
        nn.init.trunc_normal_(self.clsW, std=0.02)
        self.n = n

    def forward(self, x):
        batch, windows = x.shape[:2]
        features = self.backbone(x.flatten(0, 1)).view(batch, windows, -1)
        hidden = self.norm(features)
        attention = torch.softmax(self.att(hidden), dim=1)
        pooled = torch.einsum("bkn,bkf->bnf", attention, hidden)
        return (pooled * self.clsW).sum(-1) + self.clsb


def load_raptor(path, arch_default, res_default, device):
    payload = torch.load(path, map_location="cpu", weights_only=False)
    arch = payload.get("arch", arch_default)
    resolution = int(payload.get("res", res_default))
    backbone = build_backbone(arch, pretrained=False)
    model = RaptorClassifier(backbone, dim=backbone.num_features)
    model.load_state_dict(payload["model"], strict=True)
    model.eval().to(device)
    del payload
    gc.collect()
    return model, resolution


def eval_centers(mask, depth, k):
    valid = np.where(mask > 0)[0]
    if len(valid) < 3:
        valid = np.arange(min(3, depth))
    low, high = int(valid.min()), int(valid.max())
    centers = [c for c in range(low + 1, high) if c - 1 >= low and c + 1 <= high]
    if not centers:
        centers = [max(1, min((low + high) // 2, depth - 2))]
    index = np.linspace(0, len(centers) - 1, k).round().astype(int)
    return [centers[i] for i in index]


def eval_windows(volume, mask, k, res):
    depth = volume.shape[0]
    centers = eval_centers(mask, depth, k)
    windows = np.empty((len(centers), 3, res, res), np.float32)
    for j, center in enumerate(centers):
        center = max(1, min(center, depth - 2))
        triple = np.stack([volume[center - 1], volume[center], volume[center + 1]], 0)
        tensor = torch.from_numpy(triple.astype(np.float32) / 255.0)
        if tensor.shape[-1] != res:
            tensor = F.interpolate(tensor[None], size=(res, res), mode="bilinear",
                                   align_corners=False)[0]
        windows[j] = tensor.numpy()
    return (torch.from_numpy(windows) - RAPTOR_MEAN) / RAPTOR_STD


@torch.no_grad()
def raptor_probs(model, windows, device):
    x = windows.unsqueeze(0).to(device, non_blocking=True)

    def run():
        return torch.sigmoid(model(x).float())[0].cpu().numpy()

    if not str(device).startswith("cuda"):
        return run()
    try:
        with torch.autocast("cuda", dtype=torch.float16):
            return run()
    except RuntimeError as error:
        with contextlib.suppress(Exception):
            with torch.cuda.device(device):
                torch.cuda.empty_cache()
        log(f"  {device} fp16 retry in fp32: {type(error).__name__}")
        return run()

## 7. Branch A inference

Windows are pooled per finding rather than uniformly. Sparse focal findings —
fracture, contusion, meniscus tears — take the max or top-2 over window positions,
because averaging a finding visible in two of sixty windows dilutes it to nothing.

In [ ]:
# ============================================================================
# 7. Branch A — DINOv2 slot ensemble
# ----------------------------------------------------------------------------
# The original computed three predictions per member (a jittered primary, a
# frontier vote and a soft fold blend), then overwrote submission.csv with the
# frontier one and deleted the rest. Only the frontier path is kept here, which
# removes the augmented forward pass entirely: same numbers, roughly half the
# GPU work when jitter was affordable.
# ============================================================================
EVAL_BATCH = 8
DINO_TARGET_POOL = {
    "Fracture": "max", "Contusion": "max", "Medial Meniscus": "max",
    "Lateral Meniscus": "max", "ACL": "top2", "MCL": "top2", "Baker's": "max",
}
BUILD_LOCK = threading.Lock()
STATE_LOCK = threading.Lock()


def window_starts(n_slice, group):
    if n_slice >= group:
        return list(range(n_slice - group + 1))
    return [g * group for g in range(max(n_slice // group, 1))]


def pool_windows(probs):
    """Mean over windows, with a per-finding override for the sparse findings."""
    values = probs.mean(0).clone()
    for target, mode in DINO_TARGET_POOL.items():
        j = TARGETS.index(target)
        if mode == "max":
            values[:, j] = probs[:, :, j].max(0).values
        elif mode.startswith("top"):
            k = min(int(mode[3:]), probs.shape[0])
            values[:, j] = probs[:, :, j].topk(k, dim=0).values.mean(0)
        else:
            raise ValueError(f"unknown pooling mode for {target}: {mode}")
    return values


def augment(images, generator=None):
    """Small affine + intensity jitter, as the members were fitted with."""
    lead = images.shape[:-3]
    x = images.reshape(-1, *images.shape[-3:]).float()
    n, device = x.shape[0], x.device
    rotation = (torch.rand(n, device=device, generator=generator) - 0.5) * 2 * (
        AUG_ROT_DEG * np.pi / 180
    )
    scale = 1.0 + torch.rand(n, device=device, generator=generator) * AUG_SCALE
    shift_x = (torch.rand(n, device=device, generator=generator) - 0.5) * 2 * AUG_SHIFT
    shift_y = (torch.rand(n, device=device, generator=generator) - 0.5) * 2 * AUG_SHIFT
    cos, sin = torch.cos(rotation) / scale, torch.sin(rotation) / scale
    theta = torch.zeros(n, 2, 3, device=device, dtype=torch.float32)
    theta[:, 0, 0], theta[:, 0, 1], theta[:, 0, 2] = cos, -sin, shift_x
    theta[:, 1, 0], theta[:, 1, 1], theta[:, 1, 2] = sin, cos, shift_y
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="border", align_corners=False)
    intensity = 1.0 + (torch.rand(n, 1, 1, 1, device=device, generator=generator) - 0.5) * 2 * (
        AUG_INTENSITY
    )
    x = (x * intensity).clamp(0, 255)
    return x.reshape(*lead, *x.shape[-3:]).to(images.dtype)


@torch.no_grad()
def dino_predict(model, cache, mask, device, spec, jitter_seed=SEED):
    starts = window_starts(cache.shape[2], spec.group)
    if not starts:
        raise ValueError("no windows to average over")
    model.eval()
    generator = torch.Generator(device=device)
    generator.manual_seed(int(jitter_seed) % (2 ** 63 - 1))
    out = []
    for begin in range(0, len(cache), EVAL_BATCH):
        end = min(begin + EVAL_BATCH, len(cache))
        slot_mask = torch.from_numpy(mask[begin:end]).to(device)
        windows = []
        for start in starts:
            rows = torch.from_numpy(
                np.ascontiguousarray(cache[begin:end, :, start:start + spec.group])
            ).to(device)
            views = [rows] + ([augment(rows, generator=generator)] if FLAGS["tta_jitter"] else [])
            probabilities = []
            for view in views:
                with torch.autocast("cuda", enabled=device.type == "cuda"):
                    logits = model(view, slot_mask, spec.img).float()
                probabilities.append(torch.sigmoid(logits))
            windows.append(torch.stack(probabilities).mean(0))
        out.append(pool_windows(torch.stack(windows)).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, len(TARGETS)), np.float32)


def dino_spec(config, index):
    rules = config.get("rules") or RULES_NATIVE
    unknown = {k: v for k, v in rules.items()
               if k not in RULES_NATIVE or v not in (RULES_NATIVE[k], RULES_LEGACY[k])}
    if unknown:
        raise WeightsError(f"members record pixel rules this pipeline cannot reproduce: {unknown}")
    dialect = "legacy" if rules.get("order") == RULES_LEGACY["order"] else "native"
    spec = PixelSpec(
        name=f"dino_g{index}", slots=SLOTS_RECOVERED, img=int(config["img"]),
        slices=int(config["slices"]), group=int(config["group"]),
        crop_mm=float(config["crop_mm"]), band=tuple(float(x) for x in config["band"]),
        rules=dialect, mode="rescale",
    )
    if [slot[0] for slot in spec.slots] != list(config["slots"]):
        raise WeightsError(
            f"members were fitted on slots {config['slots']} but this pipeline defines "
            f"{[s[0] for s in spec.slots]}; a weight would be read against the wrong slot"
        )
    return spec


def _run_member(member, device, spec, cache, mask):
    started = time.time()
    with BUILD_LOCK:
        payload = torch.load(DINO_PACKAGE / member["file"], map_location="cpu", weights_only=False)
        config = member["config"]
        model = build_dino_model(
            int(config["unfreeze_last"]), spec.n_slot, variant=config["variant"],
            pool=config.get("pool", "cls_mean"), prior=bool(config.get("prior", False)),
        ).to(device)
        model.load_state_dict(payload["model"])
        stored = payload.get("fingerprint")
        if stored is not None:
            check_fingerprint(model, device, spec, stored, tag=f"{member['id']}: ")
        else:
            log(f"  {member['id']}: no stored fingerprint (legacy bundle)")
    jitter_seed = SEED + int(hashlib.sha256(str(member["id"]).encode()).hexdigest()[:8], 16)
    prediction = dino_predict(model, cache, mask, device, spec, jitter_seed=jitter_seed)
    del model, payload
    gc.collect()
    if device.type == "cuda":
        with torch.cuda.device(device):
            torch.cuda.empty_cache()
    log(f"  {member['id']} done on {device} in {time.time() - started:.0f}s")
    return prediction


def combine_rank_mean(members):
    """Weighted rank mean, then a final rank — the original _combine chain."""
    ids = sorted({study for member in members for study in member["ids"]})
    position = {study: i for i, study in enumerate(ids)}
    accumulated = np.zeros((len(ids), len(TARGETS)), np.float64)
    total = np.zeros(len(TARGETS), np.float64)
    for member in members:
        weights = np.asarray(
            member.get("target_weight") or [float(member.get("weight", 1.0))] * len(TARGETS),
            np.float64,
        )
        if weights.shape != (len(TARGETS),) or np.any(weights < 0):
            raise ValueError(f"invalid target weights for {member.get('id')}: {weights}")
        rows = [position[study] for study in member["ids"]]
        accumulated[rows] += rank_avg(member["pred"]) * weights[None, :]
        total += weights
    if np.any(total <= 0):
        raise ValueError(f"at least one target has no ensemble vote: {total}")
    return ids, accumulated / total[None, :]


def run_dino_branch():
    manifest = json.loads((DINO_PACKAGE / "manifest.json").read_text())
    members = manifest["members"]
    groups = {}
    for member in members:
        groups.setdefault(member["pixel_group"], []).append(member)
    log(f"branch A: {len(members)} member(s) in {len(groups)} pixel group(s)")

    banked = []
    for group_index, (key, group_members) in enumerate(groups.items(), 1):
        spec = dino_spec(json.loads(key), group_index)
        log(f"branch A group {group_index}: {spec.img}px x {spec.slices} slices, "
            f"crop {spec.crop_mm} mm, {len(group_members)} member(s)")
        studies, cache, mask = build_caches([spec], SERIES_FRAME, f"A/g{group_index}")[spec.name]
        pending = list(group_members)

        def worker(device):
            while True:
                with STATE_LOCK:
                    member = pending.pop(0) if pending else None
                if member is None:
                    return
                try:
                    prediction = _run_member(member, device, spec, cache, mask)
                except Exception as exc:
                    log(f"  MEMBER {member['id']} failed on {device} "
                        f"({type(exc).__name__}: {exc}); dropped — costs one vote, not the run")
                    if device.type == "cuda":
                        with torch.cuda.device(device):
                            torch.cuda.empty_cache()
                    continue
                if float(np.std(prediction)) < 1e-09:
                    log(f"  {member['id']}: degenerate predictions; not banked")
                    continue
                with STATE_LOCK:
                    banked.append({"id": member["id"], "fold": member.get("fold"),
                                   "ids": studies, "pred": prediction})

        threads = [threading.Thread(target=worker, args=(device,)) for device in DEVICES]
        for thread in threads:
            thread.start()
        for thread in threads:
            thread.join()
        cache = mask = None
        gc.collect()

    if not banked:
        raise WeightsError("no member produced predictions")
    ids, accumulated = combine_rank_mean(banked)
    log(f"branch A: rank mean of {len(banked)}/{len(members)} member(s)")
    return align_to_test(rank_avg(accumulated), ids)


STAGE["dino"] = run_dino_branch()
write_submission(STAGE["dino"], tag="branch A checkpoint")

## 8. Branch B inference

Decoding runs in a process pool while the GPU scores the previous chunk. Workers
receive resolved file paths from the cell-2 index, so they never re-read a header.

In [ ]:
# ============================================================================
# 8. Branch B — fold ensemble over 16-slice slot volumes
# ----------------------------------------------------------------------------
# Workers now receive resolved file paths instead of a directory, so the pool
# no longer re-reads every slice header that cell 2 already indexed.
# ============================================================================
FOLD_CHUNK, FOLD_MICRO = 48, 8
FOLD_WORKERS = max(1, min(4, os.cpu_count() or 4))


def fold_normalise(images, config):
    kind = config.get("norm", "none")
    if kind == "zscore":
        present = (images > 0).float()
        count = present.sum(dim=(1, 2, 3), keepdim=True).clamp(min=1.0)
        mean = (images * present).sum(dim=(1, 2, 3), keepdim=True) / count
        variance = (((images - mean) * present) ** 2).sum(dim=(1, 2, 3), keepdim=True) / count
        return (images - mean) / (variance.sqrt() + 1e-06) * present
    if kind == "imagenet":
        present = (images > 0).float()
        return (images - 0.485) / 0.229 * present
    return images


@torch.no_grad()
def fold_micro(models, config, volumes, masks, device, amp_dtype, amp_on):
    stacks, slots, segments, valids = [], [], [], []
    for row in range(len(masks)):
        present = np.nonzero(masks[row] > 0)[0]
        if len(present) == 0:
            continue
        block = volumes[row][present]
        stacks.append(torch.from_numpy(block))
        valids.append(torch.from_numpy(
            block.reshape(block.shape[0], block.shape[1], -1).max(2) > 0
        ))
        slots.append(torch.from_numpy(present + 1).long())
        segments.append(torch.full((len(present),), row, dtype=torch.long))
    out = np.full((len(models), len(masks), len(TARGETS)), np.nan, np.float32)
    if not stacks:
        return out
    images = fold_normalise(
        torch.cat(stacks).to(device, non_blocking=True).float().div_(255.0), config
    )
    slot = torch.cat(slots).to(device)
    segment = torch.cat(segments).to(device)
    valid = torch.cat(valids).to(device)
    meta = torch.zeros(len(slot), config.get("n_meta", 0), device=device)
    predictions = torch.zeros(len(models), len(masks), len(TARGETS), device=device,
                              dtype=torch.float32)
    with torch.autocast("cuda" if str(device).startswith("cuda") else "cpu",
                        dtype=amp_dtype, enabled=amp_on):
        for fold_index, model in enumerate(models):
            predictions[fold_index] = torch.sigmoid(
                model(images, slot, meta, segment, len(masks), valid=valid).float()
            )
    got = predictions.cpu().numpy()
    keep = np.array([(masks[row] > 0).any() for row in range(len(masks))])
    out[:, keep] = got[:, keep]
    return out


def run_fold_branch():
    device = DEVICES[0]
    models, config = load_fold_models()
    models = [model.to(device).eval() for model in models]
    amp_dtype, amp_on = ((torch.bfloat16, True) if str(device).startswith("cuda")
                         else (torch.float32, False))
    log(f"branch B: {len(models)} folds | norm {config.get('norm', 'none')} | "
        f"{FOLD_WORKERS} decode worker(s)")

    predictions = np.full((len(models), len(TEST_IDS), len(TARGETS)), np.nan, np.float32)
    started = time.time()
    with ProcessPoolExecutor(max_workers=FOLD_WORKERS) as pool:
        for begin in range(0, len(TEST_IDS), FOLD_CHUNK):
            block = TEST_IDS[begin:begin + FOLD_CHUNK]
            volumes = np.zeros((len(block), len(FOLD_SLOTS), FOLD_SLICES, FOLD_SIZE, FOLD_SIZE),
                               np.uint8)
            masks = np.zeros((len(block), len(FOLD_SLOTS)), np.uint8)
            futures = [pool.submit(fold_build, (i, fold_plan(study)))
                       for i, study in enumerate(block)]
            for future in as_completed(futures):
                try:
                    index, volume, mask = future.result()
                    volumes[index], masks[index] = volume, mask
                except Exception as exc:
                    log(f"  study failed: {type(exc).__name__}: {exc}")
            for micro in range(0, len(block), FOLD_MICRO):
                stop = min(micro + FOLD_MICRO, len(block))
                predictions[:, begin + micro:begin + stop] = fold_micro(
                    models, config, volumes[micro:stop], masks[micro:stop],
                    device, amp_dtype, amp_on,
                )
            done = begin + len(block)
            elapsed = time.time() - started
            log(f"  {done:,}/{len(TEST_IDS):,} | {elapsed / 60:.1f}m | "
                f"eta {elapsed / done * (len(TEST_IDS) - done) / 60:.1f}m")
            del volumes, masks
            gc.collect()

    complete = np.isfinite(predictions).all(axis=(0, 2))
    ranks = np.zeros((len(TEST_IDS), len(TARGETS)), np.float64)
    for fold_index in range(predictions.shape[0]):
        ranks[complete] += rank_ordinal(predictions[fold_index][complete])
    ranks /= predictions.shape[0]
    ranks[~complete] = np.nan
    log(f"branch B: {int(complete.sum())}/{len(TEST_IDS)} studies scored by all folds")
    for model in models:
        model.cpu()
    del models
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return ranks


STAGE["folds"] = run_fold_branch()

## 9. Branch C inference

Three slot views over the same studies: fat-saturated only (uncropped), fat-saturated
plus a structural sagittal, and the non-fat-saturated planes. The views disagree in
useful ways, which is the point of keeping all three.

In [ ]:
# ============================================================================
# 9. Branch C — RadImageNet ResNet-50 features under three slot views
# ----------------------------------------------------------------------------
# The three views (e10, e13, e11) share the fat-saturated series and used to
# be built by three independent sweeps of the whole DICOM store. build_caches
# fills whichever of them fit in memory together from one sweep.
# ============================================================================
RAD_SPECS = [
    PixelSpec("rad_e10", SLOTS_RAD_E10, img=224, slices=8, group=8, crop_mm=10000.0,
              rules="legacy", mode="rescale"),
    PixelSpec("rad_e13", SLOTS_RAD_E13, img=224, slices=8, group=8, crop_mm=130.0,
              rules="legacy", mode="rescale"),
    PixelSpec("rad_e11", SLOTS_RAD_E11, img=224, slices=8, group=8, crop_mm=130.0,
              rules="legacy", mode="rescale"),
]
RAD_COVERAGE = {"rad_e10": 0.85, "rad_e13": 0.85, "rad_e11": 0.55}
RAD_E13_MEMBER_WEIGHT = TUNING["rad_e13_w"]
RAD_V15_CONTRACT = {
    "version": "v52-radimagenet-resnet50-official-1", "targets": TARGETS,
    "encoder_sha256": RAD_ENCODER_SHA,
    "encoder_source_commit": "0ce16f7375db4236e646829d1eca61cdb4282133",
    "img": 224, "slices_per_plane": 8, "feature": "global_average_pool",
}
RAD_E13_CONTRACT = {
    "version": "e11-radimagenet-resnet50-diverse-1", "targets": TARGETS,
    "encoder_sha256": RAD_ENCODER_SHA,
    "slots": [list(slot) for slot in SLOTS_RAD_E13], "crop_mm": 130.0, "img": 224,
    "slices_per_plane": 8, "feature": "global_average_pool",
}


def rad_plan_sweeps(specs):
    """Group specs into as few sweeps as memory allows (largest group first)."""
    budget = max(available_gb() * CACHE_FRACTION, 4.0) * 1024 ** 3
    sweeps, current, used = [], [], 0.0
    for spec in specs:
        cost = len(TEST_IDS) * spec.nbytes_per_study
        if current and used + cost > budget:
            sweeps.append(current)
            current, used = [], 0.0
        current.append(spec)
        used += cost
    if current:
        sweeps.append(current)
    log(f"branch C: {len(specs)} view(s) in {len(sweeps)} sweep(s) "
        f"({budget / 1024 ** 3:.0f} GB budget)")
    return sweeps


def rad_build_views(specs, tag):
    """Cache each view and reorder it onto the submission's study order."""
    built = build_caches(specs, SERIES_FRAME, tag)
    views = {}
    for spec in specs:
        studies, pixels, masks = built[spec.name]
        position = {str(uid): i for i, uid in enumerate(studies)}
        missing = [uid for uid in TEST_IDS if uid not in position]
        if missing:
            raise WeightsError(f"{len(missing)} studies absent from {spec.name}")
        order = np.asarray([position[uid] for uid in TEST_IDS], np.int64)
        pixels, masks = pixels[order], masks[order]
        tokens = int(np.repeat(masks[:, :, None], spec.slices, axis=2).sum())
        floor = int(RAD_COVERAGE[spec.name] * len(TEST_IDS) * spec.n_slot * spec.slices)
        if tokens < floor:
            raise WeightsError(f"insufficient slices for {spec.name}: {tokens} < {floor}")
        views[spec.name] = (spec, pixels, masks)
    return views


def run_rad_branch():
    device = DEVICES[0]
    encoder = RadEncoder()
    encoder.load_state_dict(
        torch.load(verified(RAD_ENCODER, RAD_ENCODER_SHA), map_location="cpu", weights_only=True),
        strict=True,
    )
    encoder.eval().to(device)
    for parameter in encoder.parameters():
        parameter.requires_grad_(False)
    if torch.cuda.device_count() > 1:
        encoder = nn.DataParallel(encoder, device_ids=list(range(torch.cuda.device_count())))

    heads_v15 = load_rad_heads(RAD_HEADS_V15, RAD_HEADS_V15_SHA, RAD_V15_CONTRACT,
                               n_slot=len(SLOTS_RAD_E10), n_slice=8, device=device,
                               weights_only=True)
    heads_e13 = load_rad_heads(RAD_HEADS_E13, RAD_HEADS_E13_SHA, RAD_E13_CONTRACT,
                               n_slot=len(SLOTS_RAD_E13), n_slice=8, device=device,
                               weights_only=False)

    probabilities = {}
    for sweep_index, sweep in enumerate(rad_plan_sweeps(RAD_SPECS), 1):
        views = rad_build_views(sweep, f"C/s{sweep_index}")
        for name, (spec, pixels, masks) in views.items():
            features, token_mask = rad_encode(encoder, pixels, masks, device)
            heads = heads_v15 if name == "rad_e10" else heads_e13
            probabilities[name] = np.mean(
                np.stack([rad_predict(head, features, token_mask, device) for head in heads]),
                axis=0,
            )
            log(f"branch C: {name} scored by {len(heads)} head(s)")
            del features, token_mask
            gc.collect()
        del views
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # The e13 view is folded into the reference vote before it meets the
    # transformer stack; the e11 view stays a separate second-pass vote.
    reference = rank_avg(
        (1.0 - RAD_E13_MEMBER_WEIGHT) * rank_avg(probabilities["rad_e10"])
        + RAD_E13_MEMBER_WEIGHT * rank_avg(probabilities["rad_e13"])
    )
    pass2 = rank_avg(probabilities["rad_e11"])
    del encoder, heads_v15, heads_e13
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return reference, pass2


STAGE["rad_reference"], STAGE["rad_pass2"] = run_rad_branch()

## 10. Branch D inference

Two spans over the same volumes — 2-98% and 6-94% of each series — read once and
scored by two checkpoints. Where the two agree almost perfectly the complement weight
is halved: correlation manages risk here, it never creates a weight.

In [ ]:
# ============================================================================
# 10. Branch D — CoAtNet Raptor over 3-slice windows
# ----------------------------------------------------------------------------
# Slice ordering and pixel spacing come from the shared index, so this branch
# no longer re-reads every DICOM header it already paid for in cell 2.
# ============================================================================
RAPTOR_PRIMARY = {
    "file": "raptor_ft_coatnet_v5_full_swa.pt",
    "arch": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k", "res": 384, "k_eval": 62,
}
RAPTOR_LEGACY = {
    "file": "raptor_ft_coatnet_v4_full.pt",
    "arch": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k", "res": 384, "k_eval": 42,
}
# The expanded MaxSpan checkpoint's documented gains sit on ACL, both menisci,
# Lateral OA and Fracture, so the older checkpoint only contributes elsewhere.
RAPTOR_COMPLEMENT_WEIGHT = TUNING["raptor_complement"]


def find_weight_file(filename, required=True):
    direct = [f"/kaggle/input/raptor-knee-arms/{filename}",
              f"/kaggle/input/raptor-knee-arms/1/{filename}",
              f"/kaggle/input/raptor-cnn336/{filename}"]
    for path in direct:
        if os.path.exists(path):
            return path
    for directory in sorted(glob.glob("/kaggle/input/*/")):
        if "competition" in directory.lower():
            continue
        hits = glob.glob(os.path.join(directory, "**", filename), recursive=True)
        if hits:
            return hits[0]
    if required:
        raise WeightsError(f"{filename} not found under /kaggle/input")
    return None


def run_raptor_branch():
    primary_device = DEVICES[0]
    primary_model, primary_res = load_raptor(
        find_weight_file(RAPTOR_PRIMARY["file"]), RAPTOR_PRIMARY["arch"],
        RAPTOR_PRIMARY["res"], primary_device,
    )
    log(f"branch D: primary {RAPTOR_PRIMARY['file']} on {primary_device}")

    legacy_path = find_weight_file(RAPTOR_LEGACY["file"], required=False)
    legacy_model, legacy_device, legacy_res = None, None, None
    if legacy_path and torch.cuda.device_count() >= 2:
        legacy_device = torch.device("cuda:1")
        try:
            legacy_model, legacy_res = load_raptor(legacy_path, RAPTOR_LEGACY["arch"],
                                                   RAPTOR_LEGACY["res"], legacy_device)
            log(f"branch D: complement {RAPTOR_LEGACY['file']} on {legacy_device}")
        except Exception as error:
            legacy_model = None
            log(f"branch D: complement disabled safely: {type(error).__name__}: {error}")
    else:
        log("branch D: complement unavailable or second GPU absent; primary checkpoint alone")

    n = len(TEST_IDS)
    primary_predictions = np.full((n, len(TARGETS)), 0.5, np.float32)
    legacy_predictions = np.full_like(primary_predictions, 0.5)
    legacy_ok = np.zeros(n, np.bool_)
    executor = ThreadPoolExecutor(max_workers=2) if legacy_model is not None else None
    started = time.time()

    for study_index, study in enumerate(TEST_IDS):
        try:
            primary_volume, primary_mask, legacy_volume, legacy_mask = raptor_volumes(study)
            primary_windows = eval_windows(primary_volume, primary_mask,
                                           RAPTOR_PRIMARY["k_eval"], primary_res)
            if legacy_model is not None:
                legacy_windows = eval_windows(legacy_volume, legacy_mask,
                                              RAPTOR_LEGACY["k_eval"], legacy_res)
                primary_future = executor.submit(raptor_probs, primary_model, primary_windows,
                                                 primary_device)
                legacy_future = executor.submit(raptor_probs, legacy_model, legacy_windows,
                                                legacy_device)
                primary_prediction = primary_future.result()
                try:
                    legacy_prediction = legacy_future.result()
                    legacy_ok[study_index] = True
                except Exception as error:
                    legacy_prediction = primary_prediction.copy()
                    log(f"  legacy study {study_index} fallback: {type(error).__name__}: {error}")
            else:
                primary_prediction = raptor_probs(primary_model, primary_windows, primary_device)
                legacy_prediction = primary_prediction.copy()
            primary_predictions[study_index] = primary_prediction
            legacy_predictions[study_index] = legacy_prediction
        except Exception as error:
            log(f"  study {study_index} {study[:16]} FALLBACK "
                f"({type(error).__name__}: {error})")
        if (study_index + 1) % 100 == 0 or study_index + 1 == n:
            log(f"  {study_index + 1}/{n} | {time.time() - started:.0f}s")

    if executor is not None:
        executor.shutdown(wait=True)
    legacy_enabled = legacy_model is not None
    del primary_model, legacy_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    primary_rank = rank_ordinal(np.clip(primary_predictions, 0, 1))
    ranks = primary_rank.copy()
    fraction = float(legacy_ok.mean()) if legacy_enabled else 0.0
    if fraction >= 0.98:
        legacy_rank = rank_ordinal(np.clip(legacy_predictions, 0, 1))
        report = []
        for target_index, target in enumerate(TARGETS):
            weight = float(RAPTOR_COMPLEMENT_WEIGHT.get(target, 0.0))
            if weight <= 0:
                continue
            correlation = float(np.corrcoef(primary_rank[:, target_index],
                                            legacy_rank[:, target_index])[0, 1])
            # Correlation only manages risk; it never introduces a new weight.
            if not np.isfinite(correlation):
                weight = 0.0
            elif correlation > 0.992:
                weight *= 0.50
            elif correlation < 0.65:
                weight *= 0.40
            if weight <= 0:
                continue
            ranks[:, target_index] = ((1.0 - weight) * primary_rank[:, target_index]
                                      + weight * legacy_rank[:, target_index])
            report.append(f"{target}=w{weight:.3f},corr={correlation:.3f}")
        ranks = rank_ordinal(ranks)
        log("branch D complement: " + "; ".join(report))
    else:
        log(f"branch D: legacy success={fraction:.3f}; primary checkpoint used as is")
    if not np.isfinite(ranks).all():
        ranks[~np.isfinite(ranks)] = 0.5
    return ranks


try:
    STAGE["raptor"] = run_raptor_branch()
except Exception as exc:
    STAGE["raptor"] = None
    log(f"branch D failed; the transformer stack stands alone: {type(exc).__name__}: {exc}")

## 11. Fusion

The whole chain in one function, in rank space. Note the two rank conventions
(pandas average-tie and ordinal argsort): the upstream pipeline uses both in specific
places and, because ties are scored, they are not interchangeable.

In [ ]:
# ============================================================================
# 11. Fusion
# ----------------------------------------------------------------------------
# The original passed state between branches through submission.csv on disk and
# re-read it three times. The chain is identical here, just carried in memory:
#
#   A --0.55/0.45--> B --0.50 (10 of 12 targets)--> C.reference
#     --0.85/0.15--> C.pass2 --0.40 on gated targets--> calibrator
#     --per-target ~0.5--> D
# ============================================================================
import base64
import zlib

RAD_ALPHA = TUNING["rad_alpha"]
RAD_EXCLUDE = TUNING["rad_exclude"]
V48_SECOND_ALPHA = TUNING["second_alpha"]
CAL_W = TUNING["cal_w"]
FOLD_BLEND_W = TUNING["fold_blend_w"]
CAL_PAYLOAD = "eNrtmk1vI8cRhv9KsJdcKKE/q6tzc4z4ZCMBcjQWhrCRDSG2ZEjaIEGQ/57n7RlRQ3KG4jqLJAcDS4o709NdXR9vvVU9/3z30+3N/bvffRuuawgxlm7eq2ePeffrpV8v/V9euvLraD3k6ilW6zn126vYd+U6eKmx9RiaF7OSx+X1weE67K7SdSgpVSauuaecUxr3rtp1LK0HyyV7y9Gmy/E6pBhTL61Zt2gWx+WtSew6JmOoM5AHutXp+ro8+ZrlsnvJOfDdfRL+Kl4zd2bqllNmga7rvrva2OzGNFuynGxpmnxrS/069hCtpt6T1dLLOQVsTbJ1fWunG2qXAX8NiU+7VK8rdqvNU89uwQwjpWyeLdTCnVy84ua9pthSjznHXLjDpZxbLe4WPRAQCT+LrSVcGGdLzNX0XKhesC2eV5uZ67lQPDlmSzUhS2+61ELtoTOyWGjdPudc73fvnj7c/Hg7ElpyzYXjdwIZ32m7X3INZ0crtvtc833ua6fyoUYUGf4H13rJpX+GK5aa5VbeWO9ye/03bLi97i8SpaSCl49nc4gdxI2p1dZyVmQnfL56jBH/j70GqSq6dyMROLHQGvCpcSnkYsyeohNErnVjbamVjs47wOpBa7BAsa61SXulDfSIwAbAkQqDmSK38WwImKK1kFJ3d11CpFqR1mPHlj1pWVAHeDfyU2hk0vGogz0GqCBbKmFMx9xIWEAbQ8I4PUvmAplQCeuij7GNXGMk3yhBsFIfEnvVE6HFYMHDtFuLzKwpya9gisaZduAkcyAvmw1ROjmd7OnBYytpOBqJjTyK4KzT0Pi0tQJYslc0nGqcNZV7dEaRvfcSewg9h4p41iwO+4CWMkNG1326VBmHFUoB7VoakzE+JAtYN/NknS9lLJ+9S5qR56Q2kLA4qgTuplGNmUJAkbXiGbrU0HCIsgtYaGOUVXbayNeykLfpGv8lulBfgiFEmx61xm3LTlrOPoa51Ng7IyJiT4tWxrE0ZmnJRnhawZsyLgjXIC9MRu0tRAgImkJ7bUyHKk1eUgIewtRjXGDJqO1mNJrHfNUty1HRW2SSoV3FAVIA//hrUXKANxhbRbEkgqAFKiCC43qOEXsPTaKtKCdkqx3/koMYsuP+mh4HrHoQtqSIw/h4DM7EpRL5zRirAcHiUDj2SUlr9fFAsElHBX/zWgJCQlw+83Qksw8Pt9+Ty0hmOBQBh9XQAr7fxjRQ2IkGTT/w8cAiBKwRc1jwcMh+WKtMgFShNaDGJWRAechNSOMUldXTynNIUPAaEKKkCv1cLFzxaowBOmX8vU8Pj1voWa5JTPFcGN4QXm+PpZPjA8QQYYp9Qab9rZiIAuLX8AA8f/jX4kHgI+BXCtBEeOTFvNPapIyC92YAEOmHypJcCSiFOpQi712M73IL8KiBX8Hz62pDNsAH8MK/rMR+tNSRIRJwAkIIY8Rn/fD2kB2V9I7BMX+KSJ/XJtNAbwFglmQZbu/90CZcMhAeZKudKAvlSBLwNwP1aA88BYpPdJRkbADGeBzWN2IOvySVELxyU0Lx0JHQHsFpEQRsCB9iO1qT3IOHGUjMDqeczX4BiIbvCjkiLD6t6G239p+gAML1KQuAdaLLdidDV6Y6uPR+9+3LT8KrlYgzAsjg7ok+VJakimvgAjn5qUFmtTGJKXGxSadg2UuLkWok4EvGJ0Hdsr+DmpVlGsUMZkxtuTI8vDAVaIsnSIDbK0DhVSpAFlu4iRtgrLYWRaArADOcFDAfErEdwBS8dWpNqHupW3o7nIukTmxlmASq4L/51ESrznqJTSkgCzslVSMncekb8659ljeyYttxK0oX9k50n3h2kDqoapToWt8S9/yO3tTXyteLu+19rkPViKkIblLnzJhJUhYENUJCtdfWKkEFe9WTsWin6azpqJFIfEwNyLNes+PZXFV3h+tIOXjb5mzIRnCLaWKuhnPtjsCVPAOwdkhQhBSMfWLVruTgwEM/WXt1FYgv5AN4IsyBiHrOSscCBjIomksgZCPFn9grqDrE9UXDQCSPfs5RXyeuQl3gwcEIdn5JzADhpG34s4uF5NQ24eh1GWISkEmYA6iFiNezYAbkKNGJhIk+l9XNXK3r7AgLT4YnRUuHicJS0NRLLG0KDrF1IbkaMzj2MoeKzAH/kTo9KsnWJe8gDZUuxs1iPi0CayABMdLIT4qQCbfENfCiAsujIsj9KMUQ1kXwXUBFZspHZm9Zj/dB/SrVy1qOJg9ApKAlmSQNpr6SjibnNgVoqZQCr8gllgsTiFAOCFSeDcYWuKOChZQXi5dPxouF5Oo6ogVTaxDwnVE8KfQFI6Zomwgb/oI4VG2Ae1MdsTQCwQuZp5ZROZPcdudWHrZRfwe4cPE1cv9L/FDuwRwGNYOttGllMRItS1K3sFDQDAwQMvgpfIzyIeQj2yTFOhomGLsBVPtAASGLqiXKnChASW/4ILUTGlftQlUVl8FDjifbKYIzLKXmco4anNPK6ZVl8MhVBs+D5YgTo8HXuIEDQDJF4wHEYL5PvEExn4PIt7x0JumDeUgjxHeFIZDt+6xN5mALCc6pMjTZ7oy8EomCUA09MTRvvuCLxyFC+sEWCGiqjE+HIAJ4g39Bi5sadSN9MJZaFWLbpubBNBvlmwrPAONJWS1DpaIgbyKuZSbKg7qZzgNKsqnwIX8RAOvINdgfRg0jPNJBWE+5xAQ9qDaAPh4nKInagjAL9otj26U+sAnbUERTF0QYjHV8j8OEA+mohtH9SLLuoUIh6uAWHI6yAw54uEsqL6zufMAAt3yW+6xSFLmKwEmYgIPlXHYbPAo3AvKovCk/0KR5m3fmKkpFz2W3ng58h7KzqVVHqVYUpfGlAJHvQ2dN7QKmXoQITggHUz9HGZhSty5smYHeSL4pFdvbXlXZcSJV7GCHKPpsePEmVAOm41WxbuW3Y+qkrhYhisbrSBqbGiGdAkBwd5gzNehxGUVhzu5JCKa2VAyL2pfYgYgUtcTwu3RKZzO2JublhfCDbO1Qqypu8SdTWO0501Z6iAAH68g21mECvjX8bZZ7yvpViqq9BlHCnhj7DFuiXm8qEwgrAkK9hpfbuU/nN/i4l7I/qQEnpR4qVHUxgDjblWuM2UZFhOpzp+apb8i4ua8gnyHrqPMHs839gK7gaaYDDtILqNTWOF8kxVYdMxlVDwE68/F8DR2CgpCSkkzkEvKY3x9GoRqjpn8ZLj62TyEoRVV1dxrV2GtNOMosPAjqSC6pm6yRKRJFM5wnixbhh6uLl9FdjElsUvX7WxR61T1gGohOkQaej27MxiR+DQjAndQ5SgL4Yb+VMKT2UncGK4se5hfeR7Jr6nygbTJcPGK7QSpXOoBJkuMXlFTYQX0aRsObYEQn22BNrDQdKwmZ1DDaZNcANxumdkGs3pZIJcaCcxBuPSlGziTf+UNZIgqmLMTWS1/XYNBxFowH5FchZb7uT5EQqervUK+Rc15pP55mBpKra6Wju5MCbZwiKOSS/HFuZyVRBFeeIoVrc35oNnVt1ARpKDaHF2BO1z3DLILOrOVZdhIG52ojyFxsVa1k6Bq+sOS7AA0upIqpiCt9Om8+1SsIoI4/ZYFKZq9tdwnhCzrVRpoCxpqSo+0ua1DNLCOMc3X19vGSZZ9znfEAMTUpymSGqYW/kpXU6FWV6+pah9OGLveaTkopOV1d3U9oWfAhMiyK46fRgPeL9LTSDAtUjj5cFN4D8S5zmaCjmaTjJ/Kvxb3MaFrHVI27QS2vRfcMCmeqcUXY+NX6652okyj19OEC3SaFrdWyeyJMxo7gPrANn17GjQ5RgtqvregNjzr1hYOOxATeos4b/IItGRxEFZC4aNruVL1ZbGx8ojpE5moLiGNavazB+baxPoXr/gK56zjI1VGbjtG8nA3Xg3SpFl2gtqzqsM/oGgb5Axj4w+XD1g48MBFrMAnR+TRxMci1+8h+uCC1e1nmC7XEWhf2zEdIw5SsCe3Tcc18XibsU/PKhJhd7a380s67RLEvQQWUw9KKjmps7qefVeub/IZQoLbK0qxnHRFP7qlOy8BURC7Fy2LDPk7N1F1VEm1lrZbSmSWVSoNk63DQXgvIohAjYcjXU7b/zI8u7RVvPiRChSqVjkiU6YQjrT1ImVAAN/WoKEp62V2aQGAdAuQKhSW5qouyIdQ4jNe5NS6I8v10hLIeqIIMqplDn71o0dYl4fEFkRZ4LmhFJNUSQqZCNlBT2fIWnKzJ9XWwG48bOzGZWEIffcVx/q23xAziBV83gf1cE4/+mvI8/EFV3QVmPW0a6rB7ZD314ploTllOSlFYqX3X4u7TJg6jmxLRWxit1FaPtArKoHfHeb3jwPmNGDqfvS+Iv9Ns14Vv6p9rfyft+HGiFqmJIQSU++rlbegPBqDumli9zoOGSptec8O6GAWqtaAgkOgqZ3P1WhQQ08aj2KGONuEIqdZetzuLFB6qQniid8HkZTnlyGsPGnA4lY5Qu1456WnbokG9IupwU0NoPhHTwRRUqynZU0HObV8QUyZuo/lb6tFhsdxb7bCoU9qWe8vLxBwj2laVzgS+bIZGMfee1Qaldl87gBZ5jjr3Y4rq2Xaf3LatohHwxzbqBKtv4d8bHnh1eUqfVRx07jc4zfzySrj8IGV9NMFX4mBKrq6FXc4Jz154/3737u7++fbxw+3Pz9N7eg0X0mHCeHfHbHrNhrCIpdXRA/LRRC4WR9sU/ZqJB46XJsJouwU1rPT+il6uKHqNAdbJ2InUJp0wVWBV7w8NuldU7uMyajZqh2N6UfeoM6ym1zLGizF6T6AnNQZiHO/KZMijZMOV2/yKQFKv0TSXq0Hb5teE9F4HLp50QKJ3OX64edZ7ie+++PLrd7t339z+5e7mx9/88Qt+f82dx5f//Omr6e8fvv/+49Pdwz0/f3/z19vH3z7x68uH++fpKhP+/Pjw/PDh4cfv+Hz86f5Jk99/93T7eHersfff/fnmh/H3y4fH8feLv9/x96ub5+l7vq9f0wj9msf8+HH6fhnDr3kMvzRGG3p8+PizVt3vaXy/ysjox5sPzx8fbxn+7cuWv7m9v3v68PFpsfH9pcWwLc1oyEI3f/7H/cPf7p7vnhZ6ev/+X/8GYIe3xg=="


def cal_protocol(ids):
    """Twelve per-study acquisition counts, in the order the calibrator was fitted."""
    frame = SERIES_CSV.copy()
    frame["StudyInstanceUID"] = frame["StudyInstanceUID"].astype(str)
    index = pd.Index([str(uid) for uid in ids], name="StudyInstanceUID")
    table = pd.DataFrame(index=index)
    table["n_series"] = frame.groupby("StudyInstanceUID").size().reindex(index).fillna(0)
    for plane in ("Sagittal", "Coronal", "Axial"):
        part = frame[frame["Anatomical_Plane"].astype(str).eq(plane)]
        table[f"n_{plane[:3]}"] = part.groupby("StudyInstanceUID").size().reindex(index).fillna(0)
    for flag in ("Fat_Suppression", "Fluid_Sensitive"):
        marked = frame[pd.to_numeric(frame[flag], errors="coerce").fillna(0) > 0]
        prefix = flag[:3]
        table[prefix] = marked.groupby("StudyInstanceUID").size().reindex(index).fillna(0)
        for plane in ("Sagittal", "Coronal", "Axial"):
            part = marked[marked["Anatomical_Plane"].astype(str).eq(plane)]
            table[f"{prefix}_{plane[:3]}"] = (
                part.groupby("StudyInstanceUID").size().reindex(index).fillna(0)
            )
    return table


def calibrate(branch, baseline_rank, reference_rank, pass2_rank):
    payload = json.loads(zlib.decompress(base64.b64decode(CAL_PAYLOAD)).decode())
    gate = set(payload["gate"])
    protocol = cal_protocol(TEST_IDS)
    if protocol.columns.tolist() != list(payload["protocol_columns"]):
        raise WeightsError("calibration protocol layout mismatch")
    mean_rank = (baseline_rank + reference_rank + pass2_rank) / 3.0
    blocks = [baseline_rank, reference_rank, pass2_rank,
              reference_rank - baseline_rank, pass2_rank - baseline_rank, mean_rank]
    for group in payload["groups"]:
        columns = [TARGETS.index(target) for target in group]
        blocks.append(mean_rank[:, columns].mean(axis=1, keepdims=True))
    blocks.append(protocol.to_numpy(np.float64))
    features = np.concatenate(blocks, axis=1)

    centre = np.asarray(payload["mean"], np.float64)
    spread = np.asarray(payload["scale"], np.float64)
    coefficients = np.asarray(payload["coef"], np.float64)
    intercept = np.asarray(payload["intercept"], np.float64)
    if features.shape[1] != 88 or coefficients.shape != (len(TARGETS), 88):
        raise WeightsError(f"calibration feature drift: x={features.shape}, "
                           f"coef={coefficients.shape}")
    spread = np.where(np.abs(spread) > 1e-8, spread, 1.0)
    adjusted = rank_avg(((features - centre) / spread) @ coefficients.T + intercept)

    values = np.asarray(branch, np.float64).copy()
    for index, target in enumerate(TARGETS):
        if target in gate:
            values[:, index] = (1.0 - CAL_W) * values[:, index] + CAL_W * adjusted[:, index]
    return rank_avg(values), gate


def fuse():
    dino = np.asarray(STAGE["dino"], np.float64)
    folds = np.asarray(STAGE["folds"], np.float64)

    # Branch B leaves NaN for any study no fold could score; those rows fall
    # back to branch A rather than aborting the run.
    incomplete = ~np.isfinite(folds).all(axis=1)
    if incomplete.any():
        folds = folds.copy()
        folds[incomplete] = dino[incomplete]
        log(f"fusion: {int(incomplete.sum())} study(ies) fell back to branch A for the fold vote")
    stage_b = (1.0 - FOLD_BLEND_W) * rank_avg(dino) + FOLD_BLEND_W * rank_avg(folds)
    write_submission(rank_avg(stage_b), tag="A+B checkpoint")

    baseline_rank = rank_avg(stage_b)
    reference_rank = np.asarray(STAGE["rad_reference"], np.float64)
    pass2_rank = np.asarray(STAGE["rad_pass2"], np.float64)

    stage_c = stage_b.copy()
    for index, target in enumerate(TARGETS):
        if target not in RAD_EXCLUDE:
            stage_c[:, index] = ((1.0 - RAD_ALPHA) * baseline_rank[:, index]
                                 + RAD_ALPHA * reference_rank[:, index])
    stage_c = rank_avg((1.0 - V48_SECOND_ALPHA) * rank_avg(stage_c)
                       + V48_SECOND_ALPHA * pass2_rank)

    calibrated = False
    try:
        stage_c, gate = calibrate(stage_c, baseline_rank, reference_rank, pass2_rank)
        calibrated = True
        log("fusion: 88-feature calibration applied to " + ", ".join(sorted(gate)))
    except Exception as exc:
        log(f"fusion: calibration skipped safely, raw transformer kept: "
            f"{type(exc).__name__}: {exc}")
    write_submission(stage_c, tag="transformer stack")

    STAGE["transformer"] = stage_c
    raptor = STAGE.get("raptor")
    if raptor is None:
        log("fusion: branch D unavailable; the transformer stack is the submission")
        return stage_c

    transformer_rank = rank_avg(stage_c)
    raptor_rank = rank_avg(raptor)
    weights = {target: TUNING["coatnet_base"] for target in TARGETS}
    if calibrated:
        weights.update(TUNING["coatnet_overrides"])
    fallback = {target: TUNING["coatnet_base"] for target in TARGETS}
    fallback.update(TUNING["coatnet_fallback"])

    for index, target in enumerate(TARGETS):
        correlation = float(np.corrcoef(transformer_rank[:, index], raptor_rank[:, index])[0, 1])
        if not np.isfinite(correlation):
            weights[target] = fallback[target]
        elif correlation > 0.992:
            weights[target] = 0.65 * weights[target] + 0.35 * fallback[target]
        elif correlation < 0.60:
            weights[target] = 0.50 * weights[target] + 0.50 * fallback[target]

    blended = np.zeros_like(transformer_rank)
    for index, target in enumerate(TARGETS):
        weight = float(weights[target])
        blended[:, index] = ((1.0 - weight) * transformer_rank[:, index]
                             + weight * raptor_rank[:, index])
    log("fusion: CoAtNet weights " + ", ".join(
        f"{target}={weights[target]:.2f}" for target in TARGETS
        if weights[target] != TUNING["coatnet_base"]
    ))
    STAGE["transformer"] = stage_c
    return rank_avg(blended)


FINAL = fuse()
STAGE["final"] = FINAL
submission = write_submission(FINAL, tag="final")
print(submission.head().to_string(index=False))
log(f"done in {(time.time() - T0) / 60:.1f} min")

## 12. Diagnostics: where the score is actually leaking

Slot coverage, tie blocks, branch agreement. If two branches sit above 0.99 mean
Spearman then the blend between them is decoration; if a study has no filled slot it
is a guaranteed AUC drag. Both deserve more attention than another weight sweep.

In [ ]:
# ============================================================================
# 12. Diagnostics
# ----------------------------------------------------------------------------
# Under macro ROC-AUC only the ordering inside each column counts. Two things
# destroy ordering: a study nobody could score (it gets a constant, and a tie
# block earns half credit against every study it ties with), and two branches
# that agree so completely that blending them cannot change anything. Both are
# measured here rather than guessed at.
# ============================================================================
import matplotlib.pyplot as plt


def coverage_table(specs):
    rows = []
    for spec in specs:
        picked = pick_slots(SERIES_FRAME, spec)
        counts = {name: 0 for name, *_ in spec.slots}
        empty = 0
        for study in TEST_IDS:
            chosen = picked.get(study, {})
            empty += len(chosen) == 0
            for name in chosen:
                counts[name] += 1
        for name, filled in counts.items():
            rows.append({"view": spec.name, "slot": name,
                         "filled": filled, "rate": filled / max(len(TEST_IDS), 1)})
        rows.append({"view": spec.name, "slot": "— studies with no slot at all —",
                     "filled": empty, "rate": empty / max(len(TEST_IDS), 1)})
    return pd.DataFrame(rows)


def tie_report(values, name):
    values = np.asarray(values, np.float64)
    largest = []
    for index, target in enumerate(TARGETS):
        _, counts = np.unique(values[:, index], return_counts=True)
        largest.append(int(counts.max()))
    worst = int(max(largest))
    log(f"{name}: largest tie block per finding, max {worst} "
        f"({worst / len(values):.1%} of studies)")
    return pd.Series(largest, index=TARGETS, name=f"{name} max tie block")


def agreement(stages):
    """Spearman agreement between branches, averaged over the twelve findings.

    A pair at 0.99+ is contributing almost nothing to the blend; a pair below
    0.6 is either genuinely complementary or one of them is broken.
    """
    names = [name for name in stages if stages[name] is not None]
    matrix = pd.DataFrame(np.eye(len(names)), index=names, columns=names)
    for i, left in enumerate(names):
        for right in names[i + 1:]:
            a, b = rank_avg(stages[left]), rank_avg(stages[right])
            value = float(np.mean([
                np.corrcoef(a[:, j], b[:, j])[0, 1] for j in range(len(TARGETS))
            ]))
            matrix.loc[left, right] = matrix.loc[right, left] = value
    return matrix


def diagnostics():
    specs = [dino_spec(json.loads(key), i) for i, key in enumerate(
        {member["pixel_group"] for member in
         json.loads((DINO_PACKAGE / "manifest.json").read_text())["members"]}, 1)]
    coverage = coverage_table(specs + RAD_SPECS)
    print(coverage.to_string(index=False))

    stages = {name: STAGE.get(name) for name in
              ("dino", "folds", "rad_reference", "rad_pass2", "raptor", "transformer", "final")}
    ties = pd.concat([tie_report(values, name) for name, values in stages.items()
                      if values is not None], axis=1)
    print("\n", ties.to_string())

    pairwise = agreement(stages)
    print("\nbranch agreement (mean Spearman over findings)\n", pairwise.round(3).to_string())

    figure, axes = plt.subplots(1, 2, figsize=(13, 4.2))
    filled = coverage[~coverage["slot"].str.startswith("—")]
    axes[0].barh(filled["view"] + " · " + filled["slot"], filled["rate"], color="#3b7dd8")
    axes[0].set_xlim(0, 1)
    axes[0].set_xlabel("share of studies with this slot filled")
    axes[0].set_title("Slot coverage — an empty slot is fed to the model as zeros")
    axes[0].invert_yaxis()

    final = np.asarray(STAGE["final"], np.float64)
    for index, target in enumerate(TARGETS):
        axes[1].plot(np.sort(final[:, index]), np.linspace(0, 1, len(final)),
                     linewidth=1, label=target)
    axes[1].set_title("Final score distribution per finding")
    axes[1].set_xlabel("blended rank")
    axes[1].set_ylabel("cumulative share of studies")
    axes[1].legend(fontsize=6, ncol=2)
    plt.tight_layout()
    plt.show()
    return coverage, ties, pairwise


COVERAGE, TIES, AGREEMENT = diagnostics()

## 13. Validating without fooling yourself

Only 58 of 4,407 training studies carry gold labels, so every offline number here is
noisy and some of those studies may be in-sample for some branches. Which is exactly
why the acceptance rule is a bootstrap win-rate on a weight *plateau*, not a delta at
a single grid point.

In [ ]:
# ============================================================================
# 13. Offline validation  (runs only in RSNA_MODE=validate)
# ----------------------------------------------------------------------------
# The leaderboard is a scarce, noisy oracle: the public split is a fraction of
# the test set, and every weight tuned against it is a weight fitted to that
# fraction. This cell scores the same pipeline on the studies that carry gold
# labels, so a candidate change is accepted on a bootstrap margin instead of a
# leaderboard bounce.
#
# Read the numbers with two caveats:
#   1. The gold set is tiny. A 0.01 AUC difference on it is noise; the
#      bootstrap win-rate below is the statistic to look at, not the delta.
#   2. Only the Raptor arm documents that the gold studies were held out. For
#      the other branches these studies may be in-sample, so absolute AUC is an
#      upper bound. Compare variants of one branch, not branches to each other.
# ============================================================================
from sklearn.metrics import roc_auc_score


def macro_auc(labels, scores):
    """Mean ROC-AUC over the findings that have both classes present."""
    per_target = {}
    for index, target in enumerate(TARGETS):
        truth = labels[:, index]
        if len(np.unique(truth[~np.isnan(truth)])) < 2:
            continue
        keep = ~np.isnan(truth)
        per_target[target] = float(roc_auc_score(truth[keep], scores[keep, index]))
    if not per_target:
        raise RuntimeError("no finding has both classes in the gold set")
    return float(np.mean(list(per_target.values()))), pd.Series(per_target, name="AUC")


def bootstrap_win(labels, challenger, incumbent, draws=2000, seed=SEED):
    """Share of resamples in which the challenger beats the incumbent."""
    rng = np.random.default_rng(seed)
    n = len(labels)
    wins, deltas = 0, []
    for _ in range(draws):
        rows = rng.integers(0, n, n)
        try:
            a, _ = macro_auc(labels[rows], challenger[rows])
            b, _ = macro_auc(labels[rows], incumbent[rows])
        except (RuntimeError, ValueError):
            continue
        deltas.append(a - b)
        wins += a > b
    return wins / max(len(deltas), 1), float(np.mean(deltas)) if deltas else float("nan")


def evaluate_stages(labels):
    rows = []
    for name in ("dino", "folds", "rad_reference", "rad_pass2", "raptor",
                 "transformer", "final"):
        values = STAGE.get(name)
        if values is None:
            continue
        scores = np.asarray(values, np.float64)
        if not np.isfinite(scores).all():
            scores = np.nan_to_num(scores, nan=0.5)
        mean, per_target = macro_auc(labels, scores)
        rows.append(pd.Series({"macro AUC": mean, **per_target}, name=name))
    return pd.DataFrame(rows)


def sweep(labels, incumbent_weight, left, right, key, grid=np.arange(0.0, 1.01, 0.05)):
    """Macro AUC of (1-w)*left + w*right across w, with the incumbent marked.

    Use it to see whether a weight sits on a plateau (safe) or on a spike
    (fitted to whichever split produced it).
    """
    left, right = rank_avg(left), rank_avg(right)
    rows = []
    for weight in grid:
        blended = (1.0 - weight) * left + weight * right
        mean, _ = macro_auc(labels, blended)
        rows.append({"weight": round(float(weight), 3), "macro AUC": mean,
                     "incumbent": abs(weight - incumbent_weight) < 1e-9})
    frame = pd.DataFrame(rows)
    best = frame.loc[frame["macro AUC"].idxmax()]
    log(f"sweep {key}: incumbent {incumbent_weight:.2f} -> "
        f"{float(frame.loc[frame['incumbent'], 'macro AUC'].iloc[0]):.4f}, "
        f"best {best['weight']:.2f} -> {best['macro AUC']:.4f}")
    return frame


if MODE != "validate":
    log("submit mode: validation skipped (set RSNA_MODE=validate to score the gold studies)")
else:
    labels = GOLD.loc[TEST_IDS].to_numpy(np.float64)
    table = evaluate_stages(labels)
    print(table.round(4).to_string())

    incumbent = np.asarray(STAGE["final"], np.float64)
    for name in ("dino", "transformer"):
        if STAGE.get(name) is None:
            continue
        rate, delta = bootstrap_win(labels, incumbent, np.asarray(STAGE[name], np.float64))
        log(f"final beats {name} in {rate:.1%} of bootstraps (mean delta {delta:+.4f})")

    # The two weights most worth questioning: they were set on a leaderboard,
    # not on held-out data.
    sweeps = {
        "A vs B": sweep(labels, TUNING["fold_blend_w"], STAGE["dino"], STAGE["folds"], "A vs B"),
        "transformer vs CoAtNet": (
            sweep(labels, TUNING["coatnet_base"], STAGE["transformer"], STAGE["raptor"],
                  "transformer vs CoAtNet")
            if STAGE.get("raptor") is not None else None
        ),
    }
    for key, frame in sweeps.items():
        if frame is not None:
            print(f"\n{key}\n", frame.round(4).to_string(index=False))

    log("accept a change only if it wins >90% of bootstraps AND sits on a plateau, "
        "not on a single-grid-point spike")

---

## Credits, licence and limits

The model weights, the preprocessing dialects and the fusion arithmetic reproduced
here are public work by others, under Apache 2.0:

| Component | Source |
|---|---|
| Scored inference graph and dual-checkpoint target fusion | Roman Tamrazov — DINOsaur V10 |
| DINO / RadImageNet reproduction assets | Tony Li — RSNA Knee reproduction assets |
| MaxSpan and WideDense Raptor checkpoints | Dread Development — Raptor datasets |
| DINOv2 ViT-S/14 | Meta Research |
| RadImageNet ResNet-50 | RadImageNet |

The contribution here is the restructure — the single header pass, the shared decode,
the removal of the mutating globals — plus the diagnostics and the offline validation
harness. If you fork this, keep the attributions above.

**Limits worth stating plainly.** This is a competition checkpoint, not a clinical
device. The labels behind most of these weights were read out of free-text radiology
reports by a language model, and report-derived labels are known to agree with the
image-derived ground truth only about 82% of the time, so a good score here is not
the same as a good radiologist. The gold-labelled subset is 58 studies, too small to
settle anything on its own.
